# iAuditAge - automatic feature extraction (self-checking)

**How to run - this is all you do:**
1. *Runtime > Change runtime type >* **T4 GPU**, then *Runtime > Run all*. Allow Google Drive access when asked.
2. If Colab disconnects (session limit, GPU quota, idle timeout), just *Run all* again. Every finished scan
   is saved to `MyDrive/iAuditAge/outputs/` and is never redone.

Scans: IXI Guys + HH (train, 5-fold CV) and eNKI (external test). IXI-IOP is excluded.
Each scan goes through DeepBrainNet and then straight into BrainAgeNeXt, so both advance together.
(It also works on a CPU runtime: it then does DeepBrainNet only and asks for a T4 at the end.)

**Safety checks - the run stops itself, logs why, and releases the Colab machine if any of these fail:**
- every downloaded file must match the MD5 of the file used on the laptop
- pre-flight: one scan is re-run fresh each session and must reproduce the laptop's result (+/-1 y, features r>=0.98)
- every scan's DeepBrainNet age is compared to the laptop's original prediction (drift watchdog)
- BrainAgeNeXt must reproduce the laptop's orientation test on IXI002, and its error must stay plausible
- 3 failed scans in a row, or more than 5% failed, stops the run
- every saved file is re-loaded and shape/NaN-checked before it counts as done

Log: `MyDrive/iAuditAge/outputs/run_log.txt`. If it stopped, the reason is in `outputs/STOPPED.txt`.

Reference run (T4, Oct 2026): all 531 scans, no safety stops, ~150 s/scan for DeepBrainNet (CPU-bound
ANTs registration) and ~5 s/scan for BrainAgeNeXt. Results: `results/transfer_head_mae.csv` in the repo.

In [ ]:
# ===== Settings (defaults are the safe choice; normally change nothing) =====
AUTO_RELEASE_RUNTIME = True   # free the Colab machine when finished or when a safety check stops the run
DRIVE_FOLDER = 'iAuditAge'

In [ ]:
# ===== 1. Install (pinned to the versions validated on the laptop) =====
import sys, os, subprocess, importlib.metadata as im

class Stop(Exception):
    pass

WATCH = ('numpy', 'scipy', 'pandas')
loaded = [p for p in WATCH if p in sys.modules]
before = {p: im.version(p) for p in WATCH}
tfv = '.'.join(im.version('tensorflow').split('.')[:2])
pkgs = ['antspyx==0.6.3', 'antspynet==0.3.2', f'tf_keras~={tfv}.0', 'monai==1.6.1', 'torchio==1.2.1',
        'git+https://github.com/FrancescoLR/MedNeXt.git@463813b1c7352edb487f43b1eecebe507429a195']
print('installing (about 2-4 min) ...', flush=True)
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-2000:], r.stderr[-4000:])
    raise Stop('pip install failed - nothing was run. Send this output.')
env = dict(os.environ, TF_USE_LEGACY_KERAS='1', TF_CPP_MIN_LOG_LEVEL='3')
chk = subprocess.run([sys.executable, '-c',
                      'import ants, antspynet, tf_keras, monai, torchio, nnunet_mednext; print("imports ok")'],
                     capture_output=True, text=True, env=env)
print(chk.stdout.strip() or chk.stderr[-3000:])
if chk.returncode:
    raise Stop('installed packages do not import - nothing was run. Send this output.')
changed = [p for p in loaded if im.version(p) != before[p]]
if changed:
    print(f'{changed} changed after already being loaded -> restarting the runtime once. Click Run all again.')
    os.kill(os.getpid(), 9)

In [ ]:
# ===== 2. Embedded data (do not edit): scan list with download URLs + MD5s, laptop reference results =====
MANIFEST_CSV = "scan_id,dataset,site,chronological_age,sex,original_pred,fname,url,md5,size\r\nIXI002,IXI,Guys,35.8001,F,36.8402,IXI002-Guys-0828-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10452/scans/T1/resources/NIfTI/files/IXI002-Guys-0828-T1.nii.gz,b7873629cbb6eebfa8264179271da09c,6943524\r\nIXI012,IXI,HH,38.7817,M,45.5745,IXI012-HH-1211-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10453/scans/T1/resources/NIfTI/files/IXI012-HH-1211-T1.nii.gz,f742444bbe54c278641aa874f3aa1f00,10120330\r\nIXI013,IXI,HH,46.7105,M,49.6464,IXI013-HH-1212-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10454/scans/T1/resources/NIfTI/files/IXI013-HH-1212-T1.nii.gz,bd742716605ed35f41dc25e77137c444,9885697\r\nIXI014,IXI,HH,34.2368,F,36.4706,IXI014-HH-1236-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10455/scans/T1/resources/NIfTI/files/IXI014-HH-1236-T1.nii.gz,43bf4ce917151320e29ada63e2d17720,8918103\r\nIXI015,IXI,HH,24.2847,M,43.0778,IXI015-HH-1258-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10456/scans/T1/resources/NIfTI/files/IXI015-HH-1258-T1.nii.gz,ab78aa8f3a7a55c0ef24ef077e449132,8273948\r\nIXI016,IXI,Guys,55.1677,M,62.2814,IXI016-Guys-0697-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10457/scans/T1/resources/NIfTI/files/IXI016-Guys-0697-T1.nii.gz,9a4e4d0de304c59ed5c1a0fd695ada00,8275772\r\nIXI017,IXI,Guys,29.0924,F,37.8828,IXI017-Guys-0698-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10458/scans/T1/resources/NIfTI/files/IXI017-Guys-0698-T1.nii.gz,b69288b9b2f84643613fd1d2f0d568bb,7583389\r\nIXI019,IXI,Guys,58.6585,M,61.1195,IXI019-Guys-0702-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10459/scans/T1/resources/NIfTI/files/IXI019-Guys-0702-T1.nii.gz,0be946ba68da9fe62c71ccbc6fb59e26,8728502\r\nIXI020,IXI,Guys,39.4661,M,42.1405,IXI020-Guys-0700-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10460/scans/T1/resources/NIfTI/files/IXI020-Guys-0700-T1.nii.gz,a4b428e47efc97ee3b8f899162aba4f3,7379209\r\nIXI021,IXI,Guys,21.5661,F,28.4001,IXI021-Guys-0703-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10461/scans/T1/resources/NIfTI/files/IXI021-Guys-0703-T1.nii.gz,5ed21dd35ecb066267dd0acd069bc485,7990210\r\nIXI022,IXI,Guys,30.6721,M,41.4289,IXI022-Guys-0701-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10462/scans/T1/resources/NIfTI/files/IXI022-Guys-0701-T1.nii.gz,7429a323c15ad6155eec0e80646567dc,7872405\r\nIXI023,IXI,Guys,33.0787,F,32.3811,IXI023-Guys-0699-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10463/scans/T1/resources/NIfTI/files/IXI023-Guys-0699-T1.nii.gz,ed1dfecc079e263cf68454348678ef42,6498396\r\nIXI024,IXI,Guys,37.6975,F,42.3122,IXI024-Guys-0705-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10464/scans/T1/resources/NIfTI/files/IXI024-Guys-0705-T1.nii.gz,6a77ffc5fc880513b82f8b6bbc3981e1,7351915\r\nIXI025,IXI,Guys,29.7796,F,38.0055,IXI025-Guys-0852-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10465/scans/T1/resources/NIfTI/files/IXI025-Guys-0852-T1.nii.gz,2f03f7e8d901a77dab33b772fac7ba87,7334175\r\nIXI026,IXI,Guys,34.4778,F,35.6849,IXI026-Guys-0696-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10466/scans/T1/resources/NIfTI/files/IXI026-Guys-0696-T1.nii.gz,a0d2425025aa57ef6e81498d8dd2def4,6446002\r\nIXI027,IXI,Guys,30.4175,M,41.7409,IXI027-Guys-0710-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10467/scans/T1/resources/NIfTI/files/IXI027-Guys-0710-T1.nii.gz,78cec0476bb88967b747cb0bee7d36f1,8130145\r\nIXI028,IXI,Guys,74.026,M,71.5589,IXI028-Guys-1038-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10468/scans/T1/resources/NIfTI/files/IXI028-Guys-1038-T1.nii.gz,51d53f975009ee689c78b08fdb0107d2,8030516\r\nIXI029,IXI,Guys,59.2225,F,61.7132,IXI029-Guys-0829-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10469/scans/T1/resources/NIfTI/files/IXI029-Guys-0829-T1.nii.gz,0d23c4663a9ca2f10f07f6b16c24acd3,8168580\r\nIXI030,IXI,Guys,31.4606,M,26.105,IXI030-Guys-0708-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10470/scans/T1/resources/NIfTI/files/IXI030-Guys-0708-T1.nii.gz,ea6f7f2a8d1f5f99fe9b9debd61311fb,7371795\r\nIXI031,IXI,Guys,31.1348,M,39.7703,IXI031-Guys-0797-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10471/scans/T1/resources/NIfTI/files/IXI031-Guys-0797-T1.nii.gz,e97302d4302a3817e45395335ba0d97a,7342027\r\nIXI033,IXI,HH,24.7639,M,27.1962,IXI033-HH-1259-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10472/scans/T1/resources/NIfTI/files/IXI033-HH-1259-T1.nii.gz,9ae4629f0181d4e497b5c1d5dde89820,8040977\r\nIXI034,IXI,HH,24.3422,F,31.2861,IXI034-HH-1260-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10473/scans/T1/resources/NIfTI/files/IXI034-HH-1260-T1.nii.gz,cc55aafbbd5ff7caa2e793847438e7d7,7346408\r\nIXI036,IXI,Guys,23.5373,M,21.6707,IXI036-Guys-0736-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10475/scans/T1/resources/NIfTI/files/IXI036-Guys-0736-T1.nii.gz,e03752c9a0af0ce46b5ab1d9dda3333b,7826868\r\nIXI037,IXI,Guys,37.2101,F,45.3382,IXI037-Guys-0704-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10476/scans/T1/resources/NIfTI/files/IXI037-Guys-0704-T1.nii.gz,6aa0e9356c28e93d0c30227e340fc2f8,7404367\r\nIXI038,IXI,Guys,21.3826,M,25.8759,IXI038-Guys-0729-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10477/scans/T1/resources/NIfTI/files/IXI038-Guys-0729-T1.nii.gz,9a1e8d4647d049e152dd9d1a94b805cf,7302968\r\nIXI039,IXI,HH,23.3484,M,28.2956,IXI039-HH-1261-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10478/scans/T1/resources/NIfTI/files/IXI039-HH-1261-T1.nii.gz,479c3a0ed32e32cf2c25795964351900,8485112\r\nIXI040,IXI,Guys,44.0931,F,41.3128,IXI040-Guys-0724-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10479/scans/T1/resources/NIfTI/files/IXI040-Guys-0724-T1.nii.gz,654c500a141305f38434da0cdcc0789b,7063154\r\nIXI041,IXI,Guys,27.3895,M,38.1911,IXI041-Guys-0706-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10480/scans/T1/resources/NIfTI/files/IXI041-Guys-0706-T1.nii.gz,56976a3d031cbdf11c41e1b70993dbbd,7668669\r\nIXI042,IXI,Guys,27.822,F,28.4919,IXI042-Guys-0725-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10481/scans/T1/resources/NIfTI/files/IXI042-Guys-0725-T1.nii.gz,eff1f5521c4e79519141dbaf5d6ad5d0,6894821\r\nIXI043,IXI,Guys,22.6475,F,19.5402,IXI043-Guys-0714-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10482/scans/T1/resources/NIfTI/files/IXI043-Guys-0714-T1.nii.gz,f75239179ba99d472eb53dfad622da31,6920362\r\nIXI044,IXI,Guys,44.8515,F,53.3961,IXI044-Guys-0712-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10483/scans/T1/resources/NIfTI/files/IXI044-Guys-0712-T1.nii.gz,34ef4912e498c8f13f926df94a224287,7433631\r\nIXI045,IXI,Guys,40.0356,F,36.4726,IXI045-Guys-0713-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10484/scans/T1/resources/NIfTI/files/IXI045-Guys-0713-T1.nii.gz,6ad3d750201eb2b75eda04491e2526d2,7336654\r\nIXI046,IXI,Guys,37.7331,F,38.0555,IXI046-Guys-0824-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10485/scans/T1/resources/NIfTI/files/IXI046-Guys-0824-T1.nii.gz,a3ed0bcb8e0165ba8dbd059656f7a3cd,6853655\r\nIXI048,IXI,HH,50.6502,M,47.029,IXI048-HH-1326-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10486/scans/T1/resources/NIfTI/files/IXI048-HH-1326-T1.nii.gz,5c67e459291b213a3468d8fab160afec,8833734\r\nIXI049,IXI,HH,31.8549,M,44.5593,IXI049-HH-1358-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10487/scans/T1/resources/NIfTI/files/IXI049-HH-1358-T1.nii.gz,12dbd0f759b871304d14f43228751656,8587861\r\nIXI050,IXI,Guys,63.1869,M,64.5747,IXI050-Guys-0711-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10488/scans/T1/resources/NIfTI/files/IXI050-Guys-0711-T1.nii.gz,d1c929f4c410b28c8bef475706ef568a,8096647\r\nIXI051,IXI,HH,25.7413,F,32.6855,IXI051-HH-1328-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10489/scans/T1/resources/NIfTI/files/IXI051-HH-1328-T1.nii.gz,3ff4f56640da6cc3ed2782c9bb7380ae,7661087\r\nIXI052,IXI,HH,33.7632,F,36.7249,IXI052-HH-1343-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10490/scans/T1/resources/NIfTI/files/IXI052-HH-1343-T1.nii.gz,b8ff468fc13119d6c4236d586b5edd1a,7845810\r\nIXI053,IXI,Guys,53.306,M,53.6432,IXI053-Guys-0727-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10491/scans/T1/resources/NIfTI/files/IXI053-Guys-0727-T1.nii.gz,75faf4f935dfd7267c02dfc687dba1d5,8876565\r\nIXI054,IXI,Guys,41.7522,F,38.425,IXI054-Guys-0707-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10492/scans/T1/resources/NIfTI/files/IXI054-Guys-0707-T1.nii.gz,5d7fcc2fe1e636570b464d21bfeafbfa,6881452\r\nIXI055,IXI,Guys,49.4045,F,53.3387,IXI055-Guys-0730-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10493/scans/T1/resources/NIfTI/files/IXI055-Guys-0730-T1.nii.gz,5d26ededd2cc46072d40ebc2dcedea48,7171348\r\nIXI056,IXI,HH,25.9384,M,24.8732,IXI056-HH-1327-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10494/scans/T1/resources/NIfTI/files/IXI056-HH-1327-T1.nii.gz,65c6fe6a6e470f5786d00def0184fa1f,7913741\r\nIXI057,IXI,HH,28.1588,F,31.1888,IXI057-HH-1342-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10495/scans/T1/resources/NIfTI/files/IXI057-HH-1342-T1.nii.gz,4ac3055584ceb978b29cd8c69d748e83,7993827\r\nIXI058,IXI,Guys,29.2786,M,40.6388,IXI058-Guys-0726-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10496/scans/T1/resources/NIfTI/files/IXI058-Guys-0726-T1.nii.gz,362ad6f9393c02d45991e2cdcc345ea2,8580159\r\nIXI059,IXI,HH,34.1383,M,31.6883,IXI059-HH-1284-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10497/scans/T1/resources/NIfTI/files/IXI059-HH-1284-T1.nii.gz,44e4b8a72ca7f243cc4d84959c6c97c0,8206889\r\nIXI060,IXI,Guys,30.6995,M,23.7593,IXI060-Guys-0709-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10498/scans/T1/resources/NIfTI/files/IXI060-Guys-0709-T1.nii.gz,b61d868f62d26f8e74f569135f475ee7,7693272\r\nIXI061,IXI,Guys,59.0034,F,51.0752,IXI061-Guys-0715-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10499/scans/T1/resources/NIfTI/files/IXI061-Guys-0715-T1.nii.gz,77661e3094a01603ef09240ca271aa1c,7999369\r\nIXI062,IXI,Guys,36.2163,M,43.6351,IXI062-Guys-0740-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10500/scans/T1/resources/NIfTI/files/IXI062-Guys-0740-T1.nii.gz,6524a9352d7d5b5fad3410e4ed8a1710,7653384\r\nIXI063,IXI,Guys,41.1225,M,45.9148,IXI063-Guys-0742-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10501/scans/T1/resources/NIfTI/files/IXI063-Guys-0742-T1.nii.gz,991c4ce59fcac0c21aa5213a486ca55c,9097971\r\nIXI064,IXI,Guys,20.0712,M,32.6279,IXI064-Guys-0743-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10502/scans/T1/resources/NIfTI/files/IXI064-Guys-0743-T1.nii.gz,4be226654dd4e870f53a765a752f601b,8791957\r\nIXI065,IXI,Guys,23.7153,F,22.5743,IXI065-Guys-0744-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10503/scans/T1/resources/NIfTI/files/IXI065-Guys-0744-T1.nii.gz,55469eccd6e69a034f9f1d1bdf977766,6983830\r\nIXI066,IXI,Guys,46.1684,F,53.0671,IXI066-Guys-0731-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10504/scans/T1/resources/NIfTI/files/IXI066-Guys-0731-T1.nii.gz,dda36889009c9335da13be24b35e77b9,7073839\r\nIXI067,IXI,HH,24.6899,M,27.1465,IXI067-HH-1356-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10505/scans/T1/resources/NIfTI/files/IXI067-HH-1356-T1.nii.gz,f5b1eef02ff70879f5603ab83263190b,8990237\r\nIXI068,IXI,Guys,24.5695,F,23.6848,IXI068-Guys-0756-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10506/scans/T1/resources/NIfTI/files/IXI068-Guys-0756-T1.nii.gz,4a772b9625a19f30fbbafb81883b67dc,7585037\r\nIXI069,IXI,Guys,26.2724,F,30.7914,IXI069-Guys-0769-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10507/scans/T1/resources/NIfTI/files/IXI069-Guys-0769-T1.nii.gz,1679cf7f0805746fab280e8c90308437,6217478\r\nIXI070,IXI,Guys,20.6982,F,27.6594,IXI070-Guys-0767-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10508/scans/T1/resources/NIfTI/files/IXI070-Guys-0767-T1.nii.gz,f6d4480d9a2d9fb6c44ef7f34c237f40,7085820\r\nIXI071,IXI,Guys,37.0021,F,31.1661,IXI071-Guys-0770-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10509/scans/T1/resources/NIfTI/files/IXI071-Guys-0770-T1.nii.gz,2251c2efa069fca60177a69fb301e989,6636316\r\nIXI072,IXI,HH,68.6023,M,67.8671,IXI072-HH-2324-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10510/scans/T1/resources/NIfTI/files/IXI072-HH-2324-T1.nii.gz,95fe570c202bad4c2baec38784d4a09d,8890993\r\nIXI073,IXI,Guys,46.3655,F,45.1633,IXI073-Guys-0755-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10511/scans/T1/resources/NIfTI/files/IXI073-Guys-0755-T1.nii.gz,f5558995cf18cbbe4cc76fa1b69fd885,6622151\r\nIXI075,IXI,Guys,35.9918,F,37.4591,IXI075-Guys-0754-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10513/scans/T1/resources/NIfTI/files/IXI075-Guys-0754-T1.nii.gz,6ed535c78883e95058fdc4875c71de48,7182343\r\nIXI076,IXI,Guys,55.8275,M,55.9404,IXI076-Guys-0753-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10514/scans/T1/resources/NIfTI/files/IXI076-Guys-0753-T1.nii.gz,845a81cef4934d6e307d526306d5fcf1,9022231\r\nIXI077,IXI,Guys,36.4791,F,36.59,IXI077-Guys-0752-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10515/scans/T1/resources/NIfTI/files/IXI077-Guys-0752-T1.nii.gz,a70ce03eafa5fa48acc98a4c3ba81d2c,7393704\r\nIXI078,IXI,Guys,28.5065,F,31.4607,IXI078-Guys-0751-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10516/scans/T1/resources/NIfTI/files/IXI078-Guys-0751-T1.nii.gz,9ca8117817fb6752e3a35f2005d1b313,8167968\r\nIXI079,IXI,HH,52.5339,F,34.3563,IXI079-HH-1388-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10517/scans/T1/resources/NIfTI/files/IXI079-HH-1388-T1.nii.gz,2610b5cc8fa1d0b4f37652bbeff440ec,7455813\r\nIXI080,IXI,HH,21.1992,F,22.1626,IXI080-HH-1341-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10518/scans/T1/resources/NIfTI/files/IXI080-HH-1341-T1.nii.gz,28f1cf99576dee02ebfb86a216a410cb,7844053\r\nIXI083,IXI,HH,30.8939,M,35.5148,IXI083-HH-1357-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10520/scans/T1/resources/NIfTI/files/IXI083-HH-1357-T1.nii.gz,b7d5807ace42f64ced7d30e93f9de86c,8738577\r\nIXI084,IXI,Guys,38.2341,F,40.8875,IXI084-Guys-0741-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10521/scans/T1/resources/NIfTI/files/IXI084-Guys-0741-T1.nii.gz,072edfa890da2e30fb173efbca430c4b,7033786\r\nIXI085,IXI,Guys,31.8494,F,36.0392,IXI085-Guys-0759-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10522/scans/T1/resources/NIfTI/files/IXI085-Guys-0759-T1.nii.gz,ce07ba258b2f3bf8e63ecc0143cea61e,7976961\r\nIXI086,IXI,Guys,30.0698,M,31.9248,IXI086-Guys-0728-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10523/scans/T1/resources/NIfTI/files/IXI086-Guys-0728-T1.nii.gz,21c665ea006c277def13599f73b4f6b5,7935016\r\nIXI087,IXI,Guys,25.6619,F,17.0808,IXI087-Guys-0768-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10524/scans/T1/resources/NIfTI/files/IXI087-Guys-0768-T1.nii.gz,f4ff5369c2cd07ad0fcf38ce939f0add,7266508\r\nIXI089,IXI,Guys,49.4045,F,44.2321,IXI089-Guys-0757-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10526/scans/T1/resources/NIfTI/files/IXI089-Guys-0757-T1.nii.gz,109a2d804597e9d90c5b8950c97084e2,7821616\r\nIXI090,IXI,Guys,41.8207,M,54.2889,IXI090-Guys-0800-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10527/scans/T1/resources/NIfTI/files/IXI090-Guys-0800-T1.nii.gz,14ed464b745fa3a1d9d264695ca9f67c,8362378\r\nIXI091,IXI,Guys,28.6133,M,34.5789,IXI091-Guys-0762-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10528/scans/T1/resources/NIfTI/files/IXI091-Guys-0762-T1.nii.gz,f8f7438408945da15a1301264a394b3a,7539671\r\nIXI092,IXI,HH,33.2402,M,35.0032,IXI092-HH-1436-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10529/scans/T1/resources/NIfTI/files/IXI092-HH-1436-T1.nii.gz,b23be7c3b692f4c03f511bd6a3211ee8,8481288\r\nIXI093,IXI,HH,26.8775,M,46.1569,IXI093-HH-1359-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10530/scans/T1/resources/NIfTI/files/IXI093-HH-1359-T1.nii.gz,39c2af23984712027479d51c9adfc3d9,8311709\r\nIXI094,IXI,HH,24.8542,M,46.782,IXI094-HH-1355-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10531/scans/T1/resources/NIfTI/files/IXI094-HH-1355-T1.nii.gz,8a43cc2509058f4f433cbfb470a50ea9,8076229\r\nIXI095,IXI,HH,24.8953,F,29.8848,IXI095-HH-1390-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10532/scans/T1/resources/NIfTI/files/IXI095-HH-1390-T1.nii.gz,11ae1c85b507a87480871784df073007,8071859\r\nIXI096,IXI,HH,26.924,M,45.2762,IXI096-HH-1391-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10533/scans/T1/resources/NIfTI/files/IXI096-HH-1391-T1.nii.gz,9190ca631911087f89a60dcf0c8f6529,8404276\r\nIXI097,IXI,HH,35.0226,F,36.9113,IXI097-HH-1619-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10534/scans/T1/resources/NIfTI/files/IXI097-HH-1619-T1.nii.gz,6ae2f10c60a7ae6a29fb80d301f8d974,7636434\r\nIXI098,IXI,Guys,29.7303,M,34.235,IXI098-Guys-0745-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10535/scans/T1/resources/NIfTI/files/IXI098-Guys-0745-T1.nii.gz,b8bbeb69c888d5e3858908ca59f6ef8e,8307893\r\nIXI099,IXI,Guys,51.7536,M,52.1897,IXI099-Guys-0748-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10536/scans/T1/resources/NIfTI/files/IXI099-Guys-0748-T1.nii.gz,bf4c287d8370acec34fdecc81abb02cf,7972456\r\nIXI100,IXI,Guys,41.5113,F,36.7632,IXI100-Guys-0747-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10537/scans/T1/resources/NIfTI/files/IXI100-Guys-0747-T1.nii.gz,c834feb8fe2fcb1b15e929aa3869cdc3,8191161\r\nIXI101,IXI,Guys,45.7741,M,51.3785,IXI101-Guys-0749-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10538/scans/T1/resources/NIfTI/files/IXI101-Guys-0749-T1.nii.gz,3c5929701514d91028380d8d818df6f7,8642584\r\nIXI102,IXI,HH,24.6215,F,28.7306,IXI102-HH-1416-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10539/scans/T1/resources/NIfTI/files/IXI102-HH-1416-T1.nii.gz,0692d7d536b8293cc33d9321ed358add,8348690\r\nIXI103,IXI,Guys,31.8084,M,38.5822,IXI103-Guys-0750-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10540/scans/T1/resources/NIfTI/files/IXI103-Guys-0750-T1.nii.gz,7ac2171164fc48f8848df6952ac46e34,7582634\r\nIXI104,IXI,HH,26.9843,M,33.9201,IXI104-HH-1450-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10541/scans/T1/resources/NIfTI/files/IXI104-HH-1450-T1.nii.gz,c58e20690ec94e719aadb80053514a59,7647924\r\nIXI105,IXI,HH,38.9021,M,38.4166,IXI105-HH-1471-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10542/scans/T1/resources/NIfTI/files/IXI105-HH-1471-T1.nii.gz,477b0188a2b66c5cad760a59646c3fb6,8316636\r\nIXI106,IXI,Guys,36.6708,M,40.3161,IXI106-Guys-0760-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10543/scans/T1/resources/NIfTI/files/IXI106-Guys-0760-T1.nii.gz,a2bcf91918e35de45e3df010f7e437ef,7635106\r\nIXI107,IXI,Guys,31.896,F,30.9775,IXI107-Guys-0761-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10544/scans/T1/resources/NIfTI/files/IXI107-Guys-0761-T1.nii.gz,e02d287841e32d40810a301631409bb7,6702137\r\nIXI108,IXI,Guys,43.1622,M,47.3391,IXI108-Guys-0865-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10545/scans/T1/resources/NIfTI/files/IXI108-Guys-0865-T1.nii.gz,e8bf6b55ad90305526887622d79c2ad5,8242790\r\nIXI109,IXI,Guys,36.4682,F,36.624,IXI109-Guys-0732-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10546/scans/T1/resources/NIfTI/files/IXI109-Guys-0732-T1.nii.gz,dd528144b2b84aa632c348851b8ce968,7666718\r\nIXI110,IXI,Guys,37.7659,M,39.9995,IXI110-Guys-0733-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10547/scans/T1/resources/NIfTI/files/IXI110-Guys-0733-T1.nii.gz,82a7b58106906005eecc0ae7168ade36,8148358\r\nIXI111,IXI,Guys,25.1855,M,25.5913,IXI111-Guys-0734-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10548/scans/T1/resources/NIfTI/files/IXI111-Guys-0734-T1.nii.gz,0c8c73d1bdce6a4a35613fe6d36dcc24,7853140\r\nIXI112,IXI,Guys,23.6988,M,30.8527,IXI112-Guys-0735-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10549/scans/T1/resources/NIfTI/files/IXI112-Guys-0735-T1.nii.gz,f44942a21635084f5363496c81a7c5a3,7693589\r\nIXI113,IXI,Guys,32.3614,F,29.92,IXI113-Guys-0776-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10550/scans/T1/resources/NIfTI/files/IXI113-Guys-0776-T1.nii.gz,e255e5fdc6725cb1d6887b36f96bcc57,6344754\r\nIXI114,IXI,Guys,65.41,F,65.7679,IXI114-Guys-0737-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10551/scans/T1/resources/NIfTI/files/IXI114-Guys-0737-T1.nii.gz,7589a51afcabbb241015ae57a3584582,7500753\r\nIXI115,IXI,Guys,72.1314,M,65.1662,IXI115-Guys-0738-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10552/scans/T1/resources/NIfTI/files/IXI115-Guys-0738-T1.nii.gz,007329ecbfef944074196ed472c41885,7753749\r\nIXI116,IXI,Guys,23.8467,M,27.2031,IXI116-Guys-0739-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10553/scans/T1/resources/NIfTI/files/IXI116-Guys-0739-T1.nii.gz,326c6fedd47b2526b2b4b940b7b9fa72,7337095\r\nIXI118,IXI,Guys,25.9274,F,30.088,IXI118-Guys-0764-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10555/scans/T1/resources/NIfTI/files/IXI118-Guys-0764-T1.nii.gz,2e4a6177535b4a8832638b658726a203,6836466\r\nIXI119,IXI,Guys,32.1697,F,32.1618,IXI119-Guys-0765-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10556/scans/T1/resources/NIfTI/files/IXI119-Guys-0765-T1.nii.gz,cbcbec8a8f92bfc309e8d6b49b0401d3,6546873\r\nIXI120,IXI,Guys,53.6701,F,48.1143,IXI120-Guys-0766-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10557/scans/T1/resources/NIfTI/files/IXI120-Guys-0766-T1.nii.gz,c4534f943a2046fe8bdd0ddde25dad87,7425819\r\nIXI121,IXI,Guys,35.0719,M,47.7102,IXI121-Guys-0772-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10558/scans/T1/resources/NIfTI/files/IXI121-Guys-0772-T1.nii.gz,bb1716ca3410c4232bb9913d5cbe4e25,8006972\r\nIXI122,IXI,Guys,22.9733,F,24.8995,IXI122-Guys-0773-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10559/scans/T1/resources/NIfTI/files/IXI122-Guys-0773-T1.nii.gz,565d84412731ca06af829e2a0b4ac53f,7560067\r\nIXI123,IXI,Guys,46.5791,F,42.5449,IXI123-Guys-0774-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10560/scans/T1/resources/NIfTI/files/IXI123-Guys-0774-T1.nii.gz,0e227715b01c2543e7f005eacaa598dd,7046435\r\nIXI126,IXI,HH,57.2211,M,56.515,IXI126-HH-1437-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10561/scans/T1/resources/NIfTI/files/IXI126-HH-1437-T1.nii.gz,c68dc82bfc30b4df9942763b1023210f,8837610\r\nIXI127,IXI,HH,29.4976,F,34.8709,IXI127-HH-1451-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10562/scans/T1/resources/NIfTI/files/IXI127-HH-1451-T1.nii.gz,65c5e2e1d50ff1ab08e39b1f35bce4d4,7548343\r\nIXI128,IXI,HH,27.3265,F,30.1562,IXI128-HH-1470-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10563/scans/T1/resources/NIfTI/files/IXI128-HH-1470-T1.nii.gz,a3b85b2c8c31e45fc669054daa8d273f,7788756\r\nIXI129,IXI,Guys,23.039,M,30.0113,IXI129-Guys-0775-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10564/scans/T1/resources/NIfTI/files/IXI129-Guys-0775-T1.nii.gz,e8db47e56a0b82edbb4771da842b362e,8449374\r\nIXI130,IXI,HH,38.7734,M,42.429,IXI130-HH-1528-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10565/scans/T1/resources/NIfTI/files/IXI130-HH-1528-T1.nii.gz,785eeabaeaa07b86e3902a3b35b5147f,9061506\r\nIXI131,IXI,HH,45.2293,F,44.1184,IXI131-HH-1527-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10566/scans/T1/resources/NIfTI/files/IXI131-HH-1527-T1.nii.gz,1c439ce7773a5092fe07541587da567d,8227080\r\nIXI132,IXI,HH,58.7159,M,53.5179,IXI132-HH-1415-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10567/scans/T1/resources/NIfTI/files/IXI132-HH-1415-T1.nii.gz,a5822cf1c75c7fa0c773c5e8bd770f4c,8359095\r\nIXI134,IXI,Guys,47.3539,M,44.3843,IXI134-Guys-0780-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10568/scans/T1/resources/NIfTI/files/IXI134-Guys-0780-T1.nii.gz,0399fcc1422cdbd662d88455fe24af95,7995476\r\nIXI135,IXI,Guys,29.0048,M,38.0926,IXI135-Guys-0779-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10569/scans/T1/resources/NIfTI/files/IXI135-Guys-0779-T1.nii.gz,b7ac54a9193aff0789157ff8f8ddbf2a,8225319\r\nIXI136,IXI,HH,35.8056,M,37.0455,IXI136-HH-1452-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10570/scans/T1/resources/NIfTI/files/IXI136-HH-1452-T1.nii.gz,43ae5c90277fa1706cd2cbdbd8808222,9034864\r\nIXI137,IXI,HH,41.2293,M,37.7005,IXI137-HH-1472-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10571/scans/T1/resources/NIfTI/files/IXI137-HH-1472-T1.nii.gz,52336039ac177558f3428ca519241c80,8362342\r\nIXI138,IXI,Guys,33.1389,F,35.5623,IXI138-Guys-0746-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10572/scans/T1/resources/NIfTI/files/IXI138-Guys-0746-T1.nii.gz,5ed2cde5741c914e4398029c8247a904,6963749\r\nIXI139,IXI,Guys,32.9363,M,41.3287,IXI139-Guys-0815-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10573/scans/T1/resources/NIfTI/files/IXI139-Guys-0815-T1.nii.gz,3ad93d79313953b14be9a463c0f82a7d,7656740\r\nIXI140,IXI,Guys,27.8357,F,22.4106,IXI140-Guys-0787-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10574/scans/T1/resources/NIfTI/files/IXI140-Guys-0787-T1.nii.gz,85c40f849fbcf33f8e07cea5520d904f,6853845\r\nIXI141,IXI,Guys,46.0068,F,52.1786,IXI141-Guys-0789-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10575/scans/T1/resources/NIfTI/files/IXI141-Guys-0789-T1.nii.gz,d4bb6291b4ffa6645adbe5b04a037bd5,7378252\r\nIXI142,IXI,Guys,48.2546,F,50.6793,IXI142-Guys-0786-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10576/scans/T1/resources/NIfTI/files/IXI142-Guys-0786-T1.nii.gz,d9565e9701bdaaa117c8cd2078b0e9eb,7654677\r\nIXI143,IXI,Guys,31.9562,F,29.2504,IXI143-Guys-0785-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10577/scans/T1/resources/NIfTI/files/IXI143-Guys-0785-T1.nii.gz,0a5deddf3000b263681a80d58373016a,7068781\r\nIXI144,IXI,Guys,29.358,F,29.1349,IXI144-Guys-0788-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10578/scans/T1/resources/NIfTI/files/IXI144-Guys-0788-T1.nii.gz,b6fd11627314e050a92fd28aa3b9cba9,6612170\r\nIXI145,IXI,Guys,49.3854,M,62.9294,IXI145-Guys-0781-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10579/scans/T1/resources/NIfTI/files/IXI145-Guys-0781-T1.nii.gz,95ba72e2816be69c27b979fdbb54620c,8379225\r\nIXI146,IXI,HH,33.5934,M,35.8267,IXI146-HH-1389-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10580/scans/T1/resources/NIfTI/files/IXI146-HH-1389-T1.nii.gz,f1edfaf05fe5234bcd82278708e3f7f0,8534834\r\nIXI148,IXI,HH,40.5558,F,50.2044,IXI148-HH-1453-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10581/scans/T1/resources/NIfTI/files/IXI148-HH-1453-T1.nii.gz,4cd48cf6dd081effa2361d049fd1b739,8465172\r\nIXI150,IXI,HH,27.0144,F,26.8949,IXI150-HH-1550-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10584/scans/T1/resources/NIfTI/files/IXI150-HH-1550-T1.nii.gz,a696210b3aead1e6648762653fe5209e,6745258\r\nIXI151,IXI,Guys,45.3881,M,62.3474,IXI151-Guys-0793-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10585/scans/T1/resources/NIfTI/files/IXI151-Guys-0793-T1.nii.gz,904f37865e5be13075e376ac9882fe33,7183839\r\nIXI153,IXI,Guys,22.4312,M,27.0268,IXI153-Guys-0782-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10586/scans/T1/resources/NIfTI/files/IXI153-Guys-0782-T1.nii.gz,ae4114740c2905600439080b21ecd6ba,7997340\r\nIXI154,IXI,Guys,24.3422,F,34.6708,IXI154-Guys-0821-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10587/scans/T1/resources/NIfTI/files/IXI154-Guys-0821-T1.nii.gz,5f3ad754d56615b5cf571e246b710cdb,7219492\r\nIXI156,IXI,Guys,59.3812,F,59.3256,IXI156-Guys-0837-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10588/scans/T1/resources/NIfTI/files/IXI156-Guys-0837-T1.nii.gz,db1cc00c35125dce339496dd4f5b607c,7165843\r\nIXI157,IXI,Guys,54.4257,F,55.8217,IXI157-Guys-0816-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10589/scans/T1/resources/NIfTI/files/IXI157-Guys-0816-T1.nii.gz,4091eac29c2484eb2a3b53f96348abdd,9157613\r\nIXI158,IXI,Guys,75.6331,F,72.9348,IXI158-Guys-0783-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10590/scans/T1/resources/NIfTI/files/IXI158-Guys-0783-T1.nii.gz,ba7ad9b457ac10087273f12d6b4764b3,7111529\r\nIXI159,IXI,HH,24.1424,F,25.8939,IXI159-HH-1549-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10591/scans/T1/resources/NIfTI/files/IXI159-HH-1549-T1.nii.gz,6050ff87441c533a7616d7c25ad3a214,7492967\r\nIXI160,IXI,HH,39.3977,F,38.2593,IXI160-HH-1637-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10592/scans/T1/resources/NIfTI/files/IXI160-HH-1637-T1.nii.gz,1f66234c5867682d7a429cdafd9b1506,6948224\r\nIXI161,IXI,HH,65.1143,F,65.0489,IXI161-HH-2533-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10593/scans/T1/resources/NIfTI/files/IXI161-HH-2533-T1.nii.gz,d64a29d766051be6231fe3d46cf989a3,8181543\r\nIXI162,IXI,HH,50.5243,M,48.8064,IXI162-HH-1548-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10594/scans/T1/resources/NIfTI/files/IXI162-HH-1548-T1.nii.gz,83764aa566cc1886c5a2890cdff514fb,8241838\r\nIXI163,IXI,HH,48.052,M,42.0622,IXI163-HH-1621-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10595/scans/T1/resources/NIfTI/files/IXI163-HH-1621-T1.nii.gz,7057bdf1e30f1a1d0d9339020dcbcc79,8462842\r\nIXI164,IXI,Guys,64.4682,M,65.669,IXI164-Guys-0844-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10582/scans/T1/resources/NIfTI/files/IXI164-Guys-0844-T1.nii.gz,2c73f9b7d2293bca1ea20cd8eeae8361,8771705\r\nIXI165,IXI,HH,24.3751,M,23.9043,IXI165-HH-1589-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10583/scans/T1/resources/NIfTI/files/IXI165-HH-1589-T1.nii.gz,ddd93ac762d5b3e00c71313d943c35d9,8576929\r\nIXI166,IXI,Guys,41.2019,F,38.7465,IXI166-Guys-0846-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10596/scans/T1/resources/NIfTI/files/IXI166-Guys-0846-T1.nii.gz,0fcfac301ef3ae650408e8db914139c8,7112054\r\nIXI167,IXI,HH,67.5346,F,64.346,IXI167-HH-1569-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10597/scans/T1/resources/NIfTI/files/IXI167-HH-1569-T1.nii.gz,3d9ba60a0dde66e9d603c2ce72e1b249,8799956\r\nIXI168,IXI,HH,43.8248,M,41.225,IXI168-HH-1607-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10598/scans/T1/resources/NIfTI/files/IXI168-HH-1607-T1.nii.gz,a058a57c83b729c544e632f1692ec408,8645654\r\nIXI169,IXI,Guys,58.768,F,59.5977,IXI169-Guys-0842-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10599/scans/T1/resources/NIfTI/files/IXI169-Guys-0842-T1.nii.gz,b3d142283f894b8dd2180ec5a925b869,7801707\r\nIXI170,IXI,Guys,31.5921,F,35.5539,IXI170-Guys-0843-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10600/scans/T1/resources/NIfTI/files/IXI170-Guys-0843-T1.nii.gz,4752f36699e524aca84ebbb64be832a2,7577463\r\nIXI172,IXI,Guys,74.9897,M,71.2208,IXI172-Guys-0982-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10601/scans/T1/resources/NIfTI/files/IXI172-Guys-0982-T1.nii.gz,ed2292cdfdbbf2b2110f4c12bf0daccf,8790485\r\nIXI173,IXI,HH,29.5743,F,33.9067,IXI173-HH-1590-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10602/scans/T1/resources/NIfTI/files/IXI173-HH-1590-T1.nii.gz,e4edc5a0186bb98abd85c979437a702f,7884879\r\nIXI174,IXI,HH,63.0965,F,66.0269,IXI174-HH-1571-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10603/scans/T1/resources/NIfTI/files/IXI174-HH-1571-T1.nii.gz,49391542c0406c6b7913e5819aadcb48,7713684\r\nIXI175,IXI,HH,72.2656,M,72.3076,IXI175-HH-1570-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10604/scans/T1/resources/NIfTI/files/IXI175-HH-1570-T1.nii.gz,12ceb49647567f97c53b1254d257bd7d,8843772\r\nIXI176,IXI,HH,24.8542,M,34.606,IXI176-HH-1604-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10605/scans/T1/resources/NIfTI/files/IXI176-HH-1604-T1.nii.gz,2310aa3610230b9c9778dae206fa43f5,9295675\r\nIXI177,IXI,Guys,34.2888,M,42.9717,IXI177-Guys-0831-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10606/scans/T1/resources/NIfTI/files/IXI177-Guys-0831-T1.nii.gz,5fc9e83cc694d256479bdaec57029849,7998023\r\nIXI178,IXI,Guys,25.4073,M,34.7616,IXI178-Guys-0778-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10607/scans/T1/resources/NIfTI/files/IXI178-Guys-0778-T1.nii.gz,11c4437c766976e318e95a5eada92ee9,7927054\r\nIXI179,IXI,Guys,30.1437,M,34.5094,IXI179-Guys-0777-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10608/scans/T1/resources/NIfTI/files/IXI179-Guys-0777-T1.nii.gz,5fb17e44f9377e5ff75d9dc384518b37,7805393\r\nIXI180,IXI,HH,42.152,F,32.3546,IXI180-HH-1605-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10609/scans/T1/resources/NIfTI/files/IXI180-HH-1605-T1.nii.gz,d99e8084799505c687b399f500eb6120,7777372\r\nIXI181,IXI,Guys,45.6975,M,44.8283,IXI181-Guys-0790-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10610/scans/T1/resources/NIfTI/files/IXI181-Guys-0790-T1.nii.gz,d6e65388a2938aadeda6b4096e46e9a4,8783730\r\nIXI182,IXI,Guys,60.9747,M,56.7749,IXI182-Guys-0792-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10611/scans/T1/resources/NIfTI/files/IXI182-Guys-0792-T1.nii.gz,afcaa372104270d0b5c742cde82109cf,7663317\r\nIXI183,IXI,Guys,30.1766,M,39.9796,IXI183-Guys-0791-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10612/scans/T1/resources/NIfTI/files/IXI183-Guys-0791-T1.nii.gz,d806b0e6aad507b8515217fb85251753,7559891\r\nIXI184,IXI,Guys,30.1191,M,30.988,IXI184-Guys-0794-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10613/scans/T1/resources/NIfTI/files/IXI184-Guys-0794-T1.nii.gz,a04275ae8fb031a77839374e6f6f8eb5,7693260\r\nIXI185,IXI,Guys,83.4579,M,78.2422,IXI185-Guys-0795-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10614/scans/T1/resources/NIfTI/files/IXI185-Guys-0795-T1.nii.gz,5f9197455d9397fda4aa9e9022541383,8078478\r\nIXI186,IXI,Guys,61.1526,F,57.469,IXI186-Guys-0796-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10615/scans/T1/resources/NIfTI/files/IXI186-Guys-0796-T1.nii.gz,f7c172b68f79498e7188cddec9dfcd56,7675177\r\nIXI188,IXI,Guys,44.0246,F,48.1162,IXI188-Guys-0798-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10616/scans/T1/resources/NIfTI/files/IXI188-Guys-0798-T1.nii.gz,1fb5097f7cc28c9479f9a997bf2d7cbe,8174504\r\nIXI189,IXI,Guys,41.7194,F,34.5767,IXI189-Guys-0799-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10617/scans/T1/resources/NIfTI/files/IXI189-Guys-0799-T1.nii.gz,9538533a9cfa4d588254d78f12495225,7191294\r\nIXI191,IXI,Guys,44.2053,M,42.5833,IXI191-Guys-0801-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10618/scans/T1/resources/NIfTI/files/IXI191-Guys-0801-T1.nii.gz,0a2a389783e5dbd9d79c5097e02ad5eb,9108872\r\nIXI192,IXI,Guys,38.9377,M,49.947,IXI192-Guys-0878-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10619/scans/T1/resources/NIfTI/files/IXI192-Guys-0878-T1.nii.gz,6db68e167cb2a4e92848980d9ccac917,7867583\r\nIXI193,IXI,Guys,55.2389,F,55.8196,IXI193-Guys-0810-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10620/scans/T1/resources/NIfTI/files/IXI193-Guys-0810-T1.nii.gz,f30ce5def4dd3557de5990831430f0fa,7255630\r\nIXI194,IXI,Guys,48.3313,F,48.2706,IXI194-Guys-0818-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10621/scans/T1/resources/NIfTI/files/IXI194-Guys-0818-T1.nii.gz,3bfff4828801adcb8238416b9e2e44e3,8034933\r\nIXI195,IXI,HH,37.8289,M,34.5565,IXI195-HH-1620-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10622/scans/T1/resources/NIfTI/files/IXI195-HH-1620-T1.nii.gz,a8f456b4465b3e8cf5bc5ea072de2d74,8650923\r\nIXI196,IXI,Guys,29.1143,F,36.6765,IXI196-Guys-0805-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10623/scans/T1/resources/NIfTI/files/IXI196-Guys-0805-T1.nii.gz,a4155995dd86089cb16beada22d442be,6949590\r\nIXI197,IXI,Guys,60.4627,F,53.1709,IXI197-Guys-0811-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10624/scans/T1/resources/NIfTI/files/IXI197-Guys-0811-T1.nii.gz,6ed8af52020b17ca045dfc3693dbc7ca,6929714\r\nIXI198,IXI,Guys,45.2567,M,46.2312,IXI198-Guys-0803-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10625/scans/T1/resources/NIfTI/files/IXI198-Guys-0803-T1.nii.gz,ccefd29e36b727b105b16dcac5871613,7789006\r\nIXI199,IXI,Guys,63.6632,M,64.4704,IXI199-Guys-0802-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10626/scans/T1/resources/NIfTI/files/IXI199-Guys-0802-T1.nii.gz,495323eccd2ee56e2ff5f924adc90713,7924755\r\nIXI200,IXI,Guys,49.9192,F,41.708,IXI200-Guys-0812-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10627/scans/T1/resources/NIfTI/files/IXI200-Guys-0812-T1.nii.gz,157ea547e572fbc5dfb348cd33c37dff,8664760\r\nIXI201,IXI,HH,22.4148,F,25.9892,IXI201-HH-1588-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10628/scans/T1/resources/NIfTI/files/IXI201-HH-1588-T1.nii.gz,658ffaf87b640395fa4df76f431f3c81,7706897\r\nIXI202,IXI,HH,20.167,M,31.3803,IXI202-HH-1526-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10629/scans/T1/resources/NIfTI/files/IXI202-HH-1526-T1.nii.gz,2eab97f4948cc5023bcf427fc05a2785,8470167\r\nIXI204,IXI,HH,74.6366,F,70.5909,IXI204-HH-1651-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10630/scans/T1/resources/NIfTI/files/IXI204-HH-1651-T1.nii.gz,3d2cba0a55bba1cc46637a8f3641fe58,8743061\r\nIXI205,IXI,HH,45.3799,M,47.4679,IXI205-HH-1649-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10631/scans/T1/resources/NIfTI/files/IXI205-HH-1649-T1.nii.gz,2ce923bc966ddb7a1f0dcafae0eb4fe9,9344037\r\nIXI206,IXI,HH,50.0233,F,40.0894,IXI206-HH-1650-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10632/scans/T1/resources/NIfTI/files/IXI206-HH-1650-T1.nii.gz,72b634a6bb5bbc3480665e5a7e34d2f5,8268613\r\nIXI207,IXI,Guys,40.3614,F,48.1241,IXI207-Guys-0809-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10633/scans/T1/resources/NIfTI/files/IXI207-Guys-0809-T1.nii.gz,4a0fc6f036b6c83e85f9445736531021,7670713\r\nIXI208,IXI,Guys,33.9959,M,39.9176,IXI208-Guys-0808-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10634/scans/T1/resources/NIfTI/files/IXI208-Guys-0808-T1.nii.gz,187955388cde67e7b57cbababbe00ef6,9773958\r\nIXI209,IXI,Guys,32.6434,M,40.1126,IXI209-Guys-0804-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10635/scans/T1/resources/NIfTI/files/IXI209-Guys-0804-T1.nii.gz,8a09f1cca532a63921a3cbd689050571,8199696\r\nIXI210,IXI,Guys,69.3251,M,64.4813,IXI210-Guys-0856-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10636/scans/T1/resources/NIfTI/files/IXI210-Guys-0856-T1.nii.gz,8ea2f1736eeb8e22208581b1a6163055,8574370\r\nIXI211,IXI,HH,55.1184,M,60.3616,IXI211-HH-1568-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10637/scans/T1/resources/NIfTI/files/IXI211-HH-1568-T1.nii.gz,9b0f7931f3513fbb6b58ac373dc36aa7,8728359\r\nIXI212,IXI,HH,59.1294,M,55.7335,IXI212-HH-1643-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10638/scans/T1/resources/NIfTI/files/IXI212-HH-1643-T1.nii.gz,baea34341e6730d04777c6b83d1d1178,8609727\r\nIXI213,IXI,HH,54.6639,F,47.5543,IXI213-HH-1642-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10639/scans/T1/resources/NIfTI/files/IXI213-HH-1642-T1.nii.gz,818e2d81dbd3b8e752b84af35deba49b,8267156\r\nIXI214,IXI,HH,24.3121,F,22.1154,IXI214-HH-1636-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10640/scans/T1/resources/NIfTI/files/IXI214-HH-1636-T1.nii.gz,a0ff4e200d30a17dc76c64649d01ea38,8069745\r\nIXI216,IXI,HH,73.0157,M,69.2038,IXI216-HH-1635-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10641/scans/T1/resources/NIfTI/files/IXI216-HH-1635-T1.nii.gz,9693ccd47fd2f184df060aa6ea95bf77,8431208\r\nIXI217,IXI,HH,57.2101,F,48.1323,IXI217-HH-1638-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10642/scans/T1/resources/NIfTI/files/IXI217-HH-1638-T1.nii.gz,89d44d9dfb6261521c1cc8c106140e4e,7738773\r\nIXI218,IXI,HH,35.4059,F,32.8561,IXI218-HH-1815-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10643/scans/T1/resources/NIfTI/files/IXI218-HH-1815-T1.nii.gz,a68292abd7a2e7a0d7563be734b4d3ac,8528016\r\nIXI219,IXI,Guys,53.0815,F,59.0728,IXI219-Guys-0894-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10644/scans/T1/resources/NIfTI/files/IXI219-Guys-0894-T1.nii.gz,38cc2d1b5653889ead33330e34d50b52,6920637\r\nIXI221,IXI,HH,48.5476,M,47.0154,IXI221-HH-1606-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10645/scans/T1/resources/NIfTI/files/IXI221-HH-1606-T1.nii.gz,7159ee92b097f6580d077e72c74084ca,8600717\r\nIXI222,IXI,Guys,36.0602,F,36.0594,IXI222-Guys-0819-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10646/scans/T1/resources/NIfTI/files/IXI222-Guys-0819-T1.nii.gz,5bd5ec79ae66a69ee9c96a473ceafd62,7159301\r\nIXI223,IXI,Guys,63.2471,M,63.2209,IXI223-Guys-0830-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10647/scans/T1/resources/NIfTI/files/IXI223-Guys-0830-T1.nii.gz,b276897c367bc84c1928fade711a7e58,8559478\r\nIXI224,IXI,Guys,36.9199,F,39.8447,IXI224-Guys-0823-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10648/scans/T1/resources/NIfTI/files/IXI224-Guys-0823-T1.nii.gz,7d01cac3cb2a9651c4c718cc64ea2eba,7378061\r\nIXI225,IXI,Guys,47.0144,F,47.9631,IXI225-Guys-0832-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10649/scans/T1/resources/NIfTI/files/IXI225-Guys-0832-T1.nii.gz,3583c9a9126e03e4cadbe8397dbfe464,6309682\r\nIXI226,IXI,HH,41.9028,F,49.5463,IXI226-HH-1618-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10650/scans/T1/resources/NIfTI/files/IXI226-HH-1618-T1.nii.gz,20f4e81cc630c581855a1531d7840f05,8728176\r\nIXI227,IXI,Guys,67.0554,F,64.2314,IXI227-Guys-0813-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10651/scans/T1/resources/NIfTI/files/IXI227-Guys-0813-T1.nii.gz,d5c802481e810ba90c68e435463bb9ce,7919210\r\nIXI229,IXI,Guys,72.3504,F,72.2414,IXI229-Guys-0980-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10653/scans/T1/resources/NIfTI/files/IXI229-Guys-0980-T1.nii.gz,5db5cca42b6b6e095caec0ebb840af49,8172847\r\nIXI236,IXI,Guys,25.7139,M,31.6157,IXI236-Guys-0817-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10659/scans/T1/resources/NIfTI/files/IXI236-Guys-0817-T1.nii.gz,52f291d16a7cbf7cfc33ed3c8fde3cef,7572797\r\nIXI237,IXI,Guys,78.204,F,75.0446,IXI237-Guys-1049-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10660/scans/T1/resources/NIfTI/files/IXI237-Guys-1049-T1.nii.gz,c22c2a0e6a486bab609aeb2a55908c2f,6923926\r\nIXI239,IXI,HH,62.9596,F,64.909,IXI239-HH-2296-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10662/scans/T1/resources/NIfTI/files/IXI239-HH-2296-T1.nii.gz,a598c8694805c5026522d86242fe2a46,8561689\r\nIXI240,IXI,Guys,39.0363,M,44.5142,IXI240-Guys-0834-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10663/scans/T1/resources/NIfTI/files/IXI240-Guys-0834-T1.nii.gz,d4bb31c00dd3e0ef6e26ba639ecedd8b,7724735\r\nIXI241,IXI,Guys,30.0068,M,33.8549,IXI241-Guys-0833-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10664/scans/T1/resources/NIfTI/files/IXI241-Guys-0833-T1.nii.gz,bbbcdd705cead77bbfe7b4c4953a6bdb,7782553\r\nIXI242,IXI,HH,31.3703,M,36.6465,IXI242-HH-1722-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10665/scans/T1/resources/NIfTI/files/IXI242-HH-1722-T1.nii.gz,36ba2b74e0a31b0c338b5931c4c88ca0,8395704\r\nIXI244,IXI,Guys,49.0431,F,41.5451,IXI244-Guys-0841-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10666/scans/T1/resources/NIfTI/files/IXI244-Guys-0841-T1.nii.gz,bebaa3a7373310266df27b84043ea7bb,6999449\r\nIXI246,IXI,Guys,42.2204,F,42.7565,IXI246-Guys-0840-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10667/scans/T1/resources/NIfTI/files/IXI246-Guys-0840-T1.nii.gz,f33c3e433b08b5687100c8bbc549fa8d,6730897\r\nIXI247,IXI,Guys,29.8179,M,41.5588,IXI247-Guys-0838-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10668/scans/T1/resources/NIfTI/files/IXI247-Guys-0838-T1.nii.gz,b6b86bbe0adc24b38cf27298a5ab7820,9601070\r\nIXI248,IXI,HH,44.8761,F,42.3001,IXI248-HH-1972-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10669/scans/T1/resources/NIfTI/files/IXI248-HH-1972-T1.nii.gz,3a554a63e546ef9432217a1a7e0b0196,7947076\r\nIXI249,IXI,Guys,68.2683,M,68.1129,IXI249-Guys-1072-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10670/scans/T1/resources/NIfTI/files/IXI249-Guys-1072-T1.nii.gz,965e8f6d0b9811a4b793aeee25528808,7826214\r\nIXI250,IXI,Guys,72.8843,F,72.1915,IXI250-Guys-0836-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10671/scans/T1/resources/NIfTI/files/IXI250-Guys-0836-T1.nii.gz,5f4893c03b5c46e4f3ed7c6217c1dc47,7469213\r\nIXI251,IXI,Guys,80.1697,F,75.3852,IXI251-Guys-1055-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10672/scans/T1/resources/NIfTI/files/IXI251-Guys-1055-T1.nii.gz,28657604c4bc66fcbfea5d9167af39d9,8427500\r\nIXI252,IXI,HH,70.5599,F,66.3978,IXI252-HH-1693-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10673/scans/T1/resources/NIfTI/files/IXI252-HH-1693-T1.nii.gz,f08d7cc2af0fe161da97875dd87b6666,8818936\r\nIXI253,IXI,HH,24.0027,F,21.8268,IXI253-HH-1694-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10674/scans/T1/resources/NIfTI/files/IXI253-HH-1694-T1.nii.gz,60961ef6c69e58dcd9b7ee7d31861466,8089752\r\nIXI254,IXI,HH,25.5825,F,29.4516,IXI254-HH-1705-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10675/scans/T1/resources/NIfTI/files/IXI254-HH-1705-T1.nii.gz,0b69c070e36077b82e574cb9735771b3,8277919\r\nIXI255,IXI,HH,57.7029,F,52.0753,IXI255-HH-1882-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10676/scans/T1/resources/NIfTI/files/IXI255-HH-1882-T1.nii.gz,d9a1feb5252c1ccf4aabaa66a119308a,7137446\r\nIXI256,IXI,HH,27.384,M,34.7673,IXI256-HH-1723-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10677/scans/T1/resources/NIfTI/files/IXI256-HH-1723-T1.nii.gz,7f3d23fa81c03877226f763885d43c83,8546928\r\nIXI257,IXI,HH,69.5469,M,76.0777,IXI257-HH-1724-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10678/scans/T1/resources/NIfTI/files/IXI257-HH-1724-T1.nii.gz,5b4796fca2fa9982d0d2a3e3efa991a7,8969344\r\nIXI258,IXI,HH,22.5708,F,27.9547,IXI258-HH-1769-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10679/scans/T1/resources/NIfTI/files/IXI258-HH-1769-T1.nii.gz,3552961e8c7c95bea140da69f4891b11,7944904\r\nIXI259,IXI,HH,59.7837,F,54.7858,IXI259-HH-1804-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10680/scans/T1/resources/NIfTI/files/IXI259-HH-1804-T1.nii.gz,7f3516f0f8b33bf2565594c9f60db7de,7906772\r\nIXI260,IXI,HH,60.9062,M,54.7537,IXI260-HH-1805-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10681/scans/T1/resources/NIfTI/files/IXI260-HH-1805-T1.nii.gz,4343595826703b354dfb4241c77d13fc,9242345\r\nIXI261,IXI,HH,34.2067,M,45.4616,IXI261-HH-1704-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10682/scans/T1/resources/NIfTI/files/IXI261-HH-1704-T1.nii.gz,a9fb4c692a9e1cbfacc81d4420224e76,9588433\r\nIXI262,IXI,HH,20.9281,F,26.3999,IXI262-HH-1861-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10683/scans/T1/resources/NIfTI/files/IXI262-HH-1861-T1.nii.gz,fafc56038ecd2d862e7aec3c29795c32,7957544\r\nIXI263,IXI,HH,25.2129,F,40.1974,IXI263-HH-1684-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10684/scans/T1/resources/NIfTI/files/IXI263-HH-1684-T1.nii.gz,011c1cf87d1e0cf2f529b0dd7df48e3e,8464330\r\nIXI264,IXI,Guys,44.8816,F,44.1149,IXI264-Guys-0854-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10685/scans/T1/resources/NIfTI/files/IXI264-Guys-0854-T1.nii.gz,6c85c6173a3863ce39121adcfd725f96,7994713\r\nIXI266,IXI,Guys,61.065,F,58.4559,IXI266-Guys-0853-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10687/scans/T1/resources/NIfTI/files/IXI266-Guys-0853-T1.nii.gz,4afdd3149476f665f62beb9071c03b3a,8414109\r\nIXI267,IXI,HH,35.447,M,41.6204,IXI267-HH-1772-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10688/scans/T1/resources/NIfTI/files/IXI267-HH-1772-T1.nii.gz,953824ae24853838c527cc2e148e0208,8458183\r\nIXI268,IXI,Guys,44.9199,F,39.4227,IXI268-Guys-0858-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10689/scans/T1/resources/NIfTI/files/IXI268-Guys-0858-T1.nii.gz,bd00295946f86fb0dcda339d84f37d32,7471156\r\nIXI269,IXI,Guys,26.5791,F,25.1437,IXI269-Guys-0839-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10690/scans/T1/resources/NIfTI/files/IXI269-Guys-0839-T1.nii.gz,2cfadfa483b7b3eb7f26a2e746fb8d16,7427936\r\nIXI270,IXI,Guys,61.0157,F,62.119,IXI270-Guys-0847-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10691/scans/T1/resources/NIfTI/files/IXI270-Guys-0847-T1.nii.gz,1b830b82ca5fb94a7fb6dc363f3af23a,7589480\r\nIXI274,IXI,HH,73.5387,F,71.6209,IXI274-HH-2294-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10692/scans/T1/resources/NIfTI/files/IXI274-HH-2294-T1.nii.gz,999212f2c47b14d8b79b47c2baff3bed,8270433\r\nIXI275,IXI,HH,24.1752,M,28.401,IXI275-HH-1803-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10693/scans/T1/resources/NIfTI/files/IXI275-HH-1803-T1.nii.gz,9389a6b88b1c5425872bf2408386f4d9,8701834\r\nIXI276,IXI,HH,20.909,M,33.3908,IXI276-HH-1840-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10694/scans/T1/resources/NIfTI/files/IXI276-HH-1840-T1.nii.gz,7b076085e69b98ef214cb23e13b1b6b1,8130123\r\nIXI277,IXI,HH,61.3361,F,63.5065,IXI277-HH-1770-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10695/scans/T1/resources/NIfTI/files/IXI277-HH-1770-T1.nii.gz,4463c64ef525a44b57a0550ed5578396,8655279\r\nIXI278,IXI,HH,20.2136,M,22.8276,IXI278-HH-1771-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10696/scans/T1/resources/NIfTI/files/IXI278-HH-1771-T1.nii.gz,68adc271366c411fbca592270059bad5,8063290\r\nIXI279,IXI,Guys,51.8823,M,48.603,IXI279-Guys-1044-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10697/scans/T1/resources/NIfTI/files/IXI279-Guys-1044-T1.nii.gz,ba14b3df6f1c5ef7818f1cc96b514e52,7792092\r\nIXI280,IXI,HH,42.5079,F,42.8892,IXI280-HH-1860-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10698/scans/T1/resources/NIfTI/files/IXI280-HH-1860-T1.nii.gz,69f7bf204c223b4492545dfa7359f3b9,7942747\r\nIXI282,IXI,HH,40.0082,M,42.9967,IXI282-HH-2025-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10699/scans/T1/resources/NIfTI/files/IXI282-HH-2025-T1.nii.gz,71adf3beee253223ee7d1960af6b1cb6,8532927\r\nIXI284,IXI,HH,63.0445,F,59.6003,IXI284-HH-2354-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10700/scans/T1/resources/NIfTI/files/IXI284-HH-2354-T1.nii.gz,e713969c0112e7a5211ae27f0315110c,8522055\r\nIXI285,IXI,Guys,40.8131,M,39.6431,IXI285-Guys-0857-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10701/scans/T1/resources/NIfTI/files/IXI285-Guys-0857-T1.nii.gz,1964011f47148698951a05a58dca6e49,7751838\r\nIXI286,IXI,Guys,54.2779,M,54.9249,IXI286-Guys-0859-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10702/scans/T1/resources/NIfTI/files/IXI286-Guys-0859-T1.nii.gz,68b35be60d11dabc61ece59d41064b52,9015609\r\nIXI287,IXI,Guys,44.9911,F,54.2575,IXI287-Guys-0863-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10703/scans/T1/resources/NIfTI/files/IXI287-Guys-0863-T1.nii.gz,1651ab98e0ff4794f13352910b7d45c4,6833395\r\nIXI288,IXI,Guys,78.5845,F,75.3845,IXI288-Guys-0879-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10704/scans/T1/resources/NIfTI/files/IXI288-Guys-0879-T1.nii.gz,4275fa83187dae9a13d6012e36324079,7955919\r\nIXI289,IXI,Guys,60.8405,M,63.1567,IXI289-Guys-0864-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10705/scans/T1/resources/NIfTI/files/IXI289-Guys-0864-T1.nii.gz,e903eff80c3b760bbaf914e69188989f,8556314\r\nIXI295,IXI,HH,50.9021,F,52.1906,IXI295-HH-1814-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10711/scans/T1/resources/NIfTI/files/IXI295-HH-1814-T1.nii.gz,972217b121a0119be175f82f68b973c0,8472914\r\nIXI296,IXI,HH,63.9726,F,61.4156,IXI296-HH-1970-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10712/scans/T1/resources/NIfTI/files/IXI296-HH-1970-T1.nii.gz,095e932c8b6b79d51f60bf4ad1d09064,7619161\r\nIXI297,IXI,Guys,51.2909,M,62.7756,IXI297-Guys-0886-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10713/scans/T1/resources/NIfTI/files/IXI297-Guys-0886-T1.nii.gz,58b7f1957de6fdb02413948da76f085e,8430934\r\nIXI298,IXI,Guys,43.4606,F,49.3927,IXI298-Guys-0861-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10714/scans/T1/resources/NIfTI/files/IXI298-Guys-0861-T1.nii.gz,22532f9d0aa2d25578efd7a523d0596c,6655628\r\nIXI299,IXI,Guys,53.4073,F,59.9981,IXI299-Guys-0893-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10715/scans/T1/resources/NIfTI/files/IXI299-Guys-0893-T1.nii.gz,521293b39c1b34e93bf2430edfc2805e,7266264\r\nIXI300,IXI,Guys,38.2779,M,42.152,IXI300-Guys-0880-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10716/scans/T1/resources/NIfTI/files/IXI300-Guys-0880-T1.nii.gz,efb81808d5652e37877f9c93175c8a6e,8368321\r\nIXI302,IXI,HH,59.5099,M,54.0144,IXI302-HH-1883-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10717/scans/T1/resources/NIfTI/files/IXI302-HH-1883-T1.nii.gz,8dd5b7922c81069cc38c3924aa6b55c9,8245738\r\nIXI304,IXI,Guys,46.59,F,48.9328,IXI304-Guys-0862-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10719/scans/T1/resources/NIfTI/files/IXI304-Guys-0862-T1.nii.gz,802902f4274e0fcf70a452978a9ce4c5,6893781\r\nIXI308,IXI,Guys,41.2047,M,35.898,IXI308-Guys-0884-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10723/scans/T1/resources/NIfTI/files/IXI308-Guys-0884-T1.nii.gz,9bc41b5ad444d2d663754048c2323c0d,8416017\r\nIXI311,IXI,Guys,61.588,M,66.6631,IXI311-Guys-0885-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10726/scans/T1/resources/NIfTI/files/IXI311-Guys-0885-T1.nii.gz,077db3d65b535056138d92313f9a31fd,7722571\r\nIXI312,IXI,Guys,59.1485,F,53.7368,IXI312-Guys-0887-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10727/scans/T1/resources/NIfTI/files/IXI312-Guys-0887-T1.nii.gz,928e7ee8ad75fbaea9f082cfbb88ddba,6980372\r\nIXI313,IXI,HH,65.8535,M,59.1312,IXI313-HH-2241-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10728/scans/T1/resources/NIfTI/files/IXI313-HH-2241-T1.nii.gz,256036ff974eae9ac0f7f29857458e0e,9184774\r\nIXI316,IXI,HH,56.9391,M,53.1725,IXI316-HH-1862-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10731/scans/T1/resources/NIfTI/files/IXI316-HH-1862-T1.nii.gz,5ad593cff8eba397f74eccde12f72ad2,9165058\r\nIXI317,IXI,Guys,67.7947,M,70.0142,IXI317-Guys-0896-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10732/scans/T1/resources/NIfTI/files/IXI317-Guys-0896-T1.nii.gz,b48e005a0a0fe870226223f7c7480776,8287939\r\nIXI318,IXI,Guys,54.486,F,48.2027,IXI318-Guys-0895-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10733/scans/T1/resources/NIfTI/files/IXI318-Guys-0895-T1.nii.gz,8665f00d1273ab630b966d96a6fcdbc5,8201498\r\nIXI319,IXI,Guys,59.4716,M,64.8017,IXI319-Guys-0901-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10734/scans/T1/resources/NIfTI/files/IXI319-Guys-0901-T1.nii.gz,9498f6a582afd4a9810510caffd288c2,7859982\r\nIXI320,IXI,Guys,29.7385,F,24.248,IXI320-Guys-0902-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10735/scans/T1/resources/NIfTI/files/IXI320-Guys-0902-T1.nii.gz,c3e4eb04fa1e771c249a675a6e959692,7183354\r\nIXI321,IXI,Guys,54.7433,F,60.478,IXI321-Guys-0903-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10736/scans/T1/resources/NIfTI/files/IXI321-Guys-0903-T1.nii.gz,a10309d015b709e151fc1cbeaf1da592,7575774\r\nIXI324,IXI,Guys,60.0794,M,63.2778,IXI324-Guys-0922-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10738/scans/T1/resources/NIfTI/files/IXI324-Guys-0922-T1.nii.gz,0b034fc81211785163551bb38812d52a,7639445\r\nIXI325,IXI,Guys,52.6653,F,49.2519,IXI325-Guys-0911-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10739/scans/T1/resources/NIfTI/files/IXI325-Guys-0911-T1.nii.gz,ef9c9a9ccf98f0ca80794cd8a0a48802,7383898\r\nIXI326,IXI,Guys,67.7399,F,72.8777,IXI326-Guys-0907-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10740/scans/T1/resources/NIfTI/files/IXI326-Guys-0907-T1.nii.gz,fb6c911d564d687648bc91cb02d4d725,8013241\r\nIXI327,IXI,HH,73.5633,M,69.5722,IXI327-HH-1999-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10741/scans/T1/resources/NIfTI/files/IXI327-HH-1999-T1.nii.gz,4abedfd17730f75e31a9b84fe1461768,8615769\r\nIXI328,IXI,HH,61.7851,F,65.5526,IXI328-HH-2295-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10742/scans/T1/resources/NIfTI/files/IXI328-HH-2295-T1.nii.gz,b2394fb80018d293a0eab83023ab4d72,8155857\r\nIXI329,IXI,HH,55.655,M,60.3142,IXI329-HH-1908-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10743/scans/T1/resources/NIfTI/files/IXI329-HH-1908-T1.nii.gz,eb79cd8c33dc2e5001fbcabea8b32f10,8811549\r\nIXI330,IXI,Guys,44.4819,M,47.4486,IXI330-Guys-0881-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10744/scans/T1/resources/NIfTI/files/IXI330-Guys-0881-T1.nii.gz,0f5417b267b723f4f52e7627bc1fe888,8334887\r\nIXI334,IXI,HH,69.2895,F,62.1633,IXI334-HH-1907-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10748/scans/T1/resources/NIfTI/files/IXI334-HH-1907-T1.nii.gz,1b754632d3a16aefd67b89330f5278ad,8466779\r\nIXI335,IXI,HH,69.2758,M,67.3678,IXI335-HH-1906-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10749/scans/T1/resources/NIfTI/files/IXI335-HH-1906-T1.nii.gz,56729267973e2a1b05b24e84e7570247,9337800\r\nIXI336,IXI,Guys,67.7728,M,63.7614,IXI336-Guys-0904-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10750/scans/T1/resources/NIfTI/files/IXI336-Guys-0904-T1.nii.gz,f1f96408f9697e4c2ec4183160ac9743,8717216\r\nIXI338,IXI,HH,61.4867,F,63.0424,IXI338-HH-1971-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10752/scans/T1/resources/NIfTI/files/IXI338-HH-1971-T1.nii.gz,e19dc8daf076daeb3168b8c84d3bf060,8568165\r\nIXI342,IXI,Guys,48.679,M,45.8539,IXI342-Guys-0909-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10755/scans/T1/resources/NIfTI/files/IXI342-Guys-0909-T1.nii.gz,882d780d380132a5146336e186a60c16,7398662\r\nIXI344,IXI,Guys,62.0945,M,61.0248,IXI344-Guys-0905-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10756/scans/T1/resources/NIfTI/files/IXI344-Guys-0905-T1.nii.gz,e5e75f7e79dfc8add91b6dc3db4e9e54,7527671\r\nIXI348,IXI,Guys,38.5873,F,39.2659,IXI348-Guys-0910-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10759/scans/T1/resources/NIfTI/files/IXI348-Guys-0910-T1.nii.gz,1416f51f5929ab6b2683e829c87e6f1f,7704840\r\nIXI350,IXI,Guys,62.6804,M,65.1239,IXI350-Guys-0908-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10760/scans/T1/resources/NIfTI/files/IXI350-Guys-0908-T1.nii.gz,584d88759d586ee72266e7dcaa1a67a6,7189023\r\nIXI351,IXI,Guys,56.9199,F,51.2914,IXI351-Guys-0914-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10761/scans/T1/resources/NIfTI/files/IXI351-Guys-0914-T1.nii.gz,159d4a7f1579cb65088ee426933d2f80,7304678\r\nIXI353,IXI,HH,52.6899,F,51.4223,IXI353-HH-1996-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10762/scans/T1/resources/NIfTI/files/IXI353-HH-1996-T1.nii.gz,7e0060c5b99f90c8bb157ba964eceeb9,8135536\r\nIXI354,IXI,HH,45.6153,M,51.1708,IXI354-HH-2024-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10763/scans/T1/resources/NIfTI/files/IXI354-HH-2024-T1.nii.gz,0cb9aad0b81348875e5e54b48b76a3c8,8862960\r\nIXI356,IXI,HH,64.104,F,52.3351,IXI356-HH-2049-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10764/scans/T1/resources/NIfTI/files/IXI356-HH-2049-T1.nii.gz,4d94a765e696ddbef6243f765216355f,8822001\r\nIXI357,IXI,HH,63.54,F,57.4131,IXI357-HH-2076-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10765/scans/T1/resources/NIfTI/files/IXI357-HH-2076-T1.nii.gz,6bbb6812e55a6a20f57a041135bc006f,8366931\r\nIXI358,IXI,Guys,62.0616,F,58.7759,IXI358-Guys-0919-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10766/scans/T1/resources/NIfTI/files/IXI358-Guys-0919-T1.nii.gz,99f74eeb2b31d46f822150c14b829d5f,8432828\r\nIXI359,IXI,Guys,62.0616,F,54.8482,IXI359-Guys-0918-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10767/scans/T1/resources/NIfTI/files/IXI359-Guys-0918-T1.nii.gz,4dc412a53b0e041f11697f5ff49f2b3d,8154523\r\nIXI360,IXI,Guys,54.193,F,46.5762,IXI360-Guys-0912-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10768/scans/T1/resources/NIfTI/files/IXI360-Guys-0912-T1.nii.gz,ab8f3de307c7de78af4346080c157f21,7883663\r\nIXI361,IXI,Guys,54.193,F,49.0158,IXI361-Guys-0913-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10769/scans/T1/resources/NIfTI/files/IXI361-Guys-0913-T1.nii.gz,95b74395850d1e1b55226d6206c78181,7480716\r\nIXI362,IXI,HH,41.0951,F,33.086,IXI362-HH-2051-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10770/scans/T1/resources/NIfTI/files/IXI362-HH-2051-T1.nii.gz,bf9a5004c40f2a1177985abf5ddd1cd3,7936731\r\nIXI363,IXI,HH,60.9199,F,58.7784,IXI363-HH-2050-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10771/scans/T1/resources/NIfTI/files/IXI363-HH-2050-T1.nii.gz,884f2d07fe98383a0d1fd9d2f16091e3,8100725\r\nIXI364,IXI,Guys,67.1622,F,58.7516,IXI364-Guys-0920-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10772/scans/T1/resources/NIfTI/files/IXI364-Guys-0920-T1.nii.gz,6e1faa9add7c8c86e5d81d9d4a0d6064,7304468\r\nIXI365,IXI,Guys,66.527,F,66.0275,IXI365-Guys-0923-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10773/scans/T1/resources/NIfTI/files/IXI365-Guys-0923-T1.nii.gz,87f972e8c6306e299d24dface9a2ac4d,7712524\r\nIXI367,IXI,Guys,54.2149,F,49.476,IXI367-Guys-0917-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10774/scans/T1/resources/NIfTI/files/IXI367-Guys-0917-T1.nii.gz,8ab92c1b116c264d55dffe003c0cf3b1,7925141\r\nIXI368,IXI,Guys,54.2149,F,51.9559,IXI368-Guys-0916-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10775/scans/T1/resources/NIfTI/files/IXI368-Guys-0916-T1.nii.gz,3c1ed277b226232fa963ee93c44a7209,7765102\r\nIXI369,IXI,Guys,60.5969,F,55.9173,IXI369-Guys-0924-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10776/scans/T1/resources/NIfTI/files/IXI369-Guys-0924-T1.nii.gz,bf39746717ed84369d4c9909f403b235,9031097\r\nIXI370,IXI,Guys,60.5969,F,62.7768,IXI370-Guys-0921-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10777/scans/T1/resources/NIfTI/files/IXI370-Guys-0921-T1.nii.gz,d5ee7845c5a20ca01a5f7e4c31e360cf,10066896\r\nIXI375,IXI,Guys,41.0595,F,38.2746,IXI375-Guys-0925-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10784/scans/T1/resources/NIfTI/files/IXI375-Guys-0925-T1.nii.gz,14824c33a155639342e69a351eb1f547,7314542\r\nIXI376,IXI,Guys,72.5941,F,68.3897,IXI376-Guys-0938-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10785/scans/T1/resources/NIfTI/files/IXI376-Guys-0938-T1.nii.gz,24f68b117b8d69faf3c42df59282ec98,6875644\r\nIXI377,IXI,Guys,72.5941,F,70.3239,IXI377-Guys-0937-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10786/scans/T1/resources/NIfTI/files/IXI377-Guys-0937-T1.nii.gz,161017ec6d08b6b0fb7ecc299a9b7a2f,7283143\r\nIXI379,IXI,Guys,66.9076,F,68.2015,IXI379-Guys-0943-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10788/scans/T1/resources/NIfTI/files/IXI379-Guys-0943-T1.nii.gz,6f2b504576ef0376a7991ae2e4084d73,7801356\r\nIXI380,IXI,Guys,66.9076,F,61.2648,IXI380-Guys-0944-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10789/scans/T1/resources/NIfTI/files/IXI380-Guys-0944-T1.nii.gz,63f6d94d3d3978fb7c54d268cbf9b536,7360598\r\nIXI381,IXI,Guys,59.6715,F,69.1486,IXI381-Guys-1024-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10790/scans/T1/resources/NIfTI/files/IXI381-Guys-1024-T1.nii.gz,a03e28effa6dfd337127631008c8afed,7951426\r\nIXI383,IXI,HH,74.0123,F,70.2105,IXI383-HH-2099-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10792/scans/T1/resources/NIfTI/files/IXI383-HH-2099-T1.nii.gz,bb930313c4d306af987b2f8c9dc8916c,8410159\r\nIXI384,IXI,HH,74.0123,F,71.6753,IXI384-HH-2100-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10793/scans/T1/resources/NIfTI/files/IXI384-HH-2100-T1.nii.gz,fa4fb5f7abea706f966e28764ccc6c0e,8169062\r\nIXI385,IXI,HH,64.6461,F,71.0807,IXI385-HH-2078-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10794/scans/T1/resources/NIfTI/files/IXI385-HH-2078-T1.nii.gz,cc30d92163ffa0f8a832da912aefcc99,9117201\r\nIXI386,IXI,HH,52.3012,F,38.0368,IXI386-HH-2077-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10795/scans/T1/resources/NIfTI/files/IXI386-HH-2077-T1.nii.gz,9c8ef90ecfdaf265fda09bd0ea055f27,7740882\r\nIXI387,IXI,HH,46.4312,F,47.6738,IXI387-HH-2101-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10796/scans/T1/resources/NIfTI/files/IXI387-HH-2101-T1.nii.gz,329403e22c7eb3db125af434868cb572,8108434\r\nIXI389,IXI,Guys,58.3682,F,48.9371,IXI389-Guys-0930-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10798/scans/T1/resources/NIfTI/files/IXI389-Guys-0930-T1.nii.gz,73ae13f92a4f7b6b64461911b50dc19c,7939641\r\nIXI390,IXI,Guys,41.8891,F,41.2,IXI390-Guys-0931-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10799/scans/T1/resources/NIfTI/files/IXI390-Guys-0931-T1.nii.gz,2f91d1d35cf0c2d2bd202d04c5b39ef5,7891948\r\nIXI391,IXI,Guys,52.5311,M,56.3909,IXI391-Guys-0934-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10800/scans/T1/resources/NIfTI/files/IXI391-Guys-0934-T1.nii.gz,50f40576c153c1c8975a501ed5e19896,7605525\r\nIXI392,IXI,Guys,62.4559,F,64.9061,IXI392-Guys-1064-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10801/scans/T1/resources/NIfTI/files/IXI392-Guys-1064-T1.nii.gz,be1bcc4bf51ba845c567ff3403f1ef66,7013368\r\nIXI393,IXI,Guys,60.1013,F,62.6996,IXI393-Guys-0941-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10802/scans/T1/resources/NIfTI/files/IXI393-Guys-0941-T1.nii.gz,13d60b4ae62e24de095a0d6ad34521ea,8178261\r\nIXI394,IXI,Guys,53.7495,M,58.8215,IXI394-Guys-0940-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10803/scans/T1/resources/NIfTI/files/IXI394-Guys-0940-T1.nii.gz,1bb1c5309487f63c3a0285528d5854e1,7996991\r\nIXI396,IXI,HH,53.5661,F,53.9193,IXI396-HH-2115-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10805/scans/T1/resources/NIfTI/files/IXI396-HH-2115-T1.nii.gz,1b375c3ae9095bc5dd47cec01846b010,8821131\r\nIXI397,IXI,Guys,51.6578,F,53.3184,IXI397-Guys-0953-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10806/scans/T1/resources/NIfTI/files/IXI397-Guys-0953-T1.nii.gz,0f6d063b701b7323c5b4c04a48603db5,7868206\r\nIXI398,IXI,Guys,51.6578,F,44.573,IXI398-Guys-0952-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10807/scans/T1/resources/NIfTI/files/IXI398-Guys-0952-T1.nii.gz,4478493906258dfd8cbe92f232245578,8023403\r\nIXI399,IXI,Guys,73.5441,F,64.6759,IXI399-Guys-0966-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10808/scans/T1/resources/NIfTI/files/IXI399-Guys-0966-T1.nii.gz,b772f5fee26faac631ab690cf3c08bf9,7754843\r\nIXI400,IXI,Guys,55.5811,F,54.2923,IXI400-Guys-0977-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10809/scans/T1/resources/NIfTI/files/IXI400-Guys-0977-T1.nii.gz,88034dd4c074999e470aa7ca89d6da94,6794927\r\nIXI401,IXI,Guys,55.5811,F,56.7599,IXI401-Guys-0978-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10810/scans/T1/resources/NIfTI/files/IXI401-Guys-0978-T1.nii.gz,fdbeb12d727024cd2f8be4d77c9b18d1,7575233\r\nIXI402,IXI,Guys,53.5359,F,44.9202,IXI402-Guys-0961-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10811/scans/T1/resources/NIfTI/files/IXI402-Guys-0961-T1.nii.gz,a833c89ece841d8c32eb9a3e815c6881,7527625\r\nIXI403,IXI,Guys,53.5359,F,40.4886,IXI403-Guys-0965-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10812/scans/T1/resources/NIfTI/files/IXI403-Guys-0965-T1.nii.gz,7cbf98ad0b780bbb9320512d20db3bcf,7899879\r\nIXI404,IXI,Guys,62.7379,M,72.9924,IXI404-Guys-0950-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10813/scans/T1/resources/NIfTI/files/IXI404-Guys-0950-T1.nii.gz,2180e92ff7e317acac5a8ee4bdfa8809,8226140\r\nIXI405,IXI,Guys,58.5681,F,59.5968,IXI405-Guys-0948-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10814/scans/T1/resources/NIfTI/files/IXI405-Guys-0948-T1.nii.gz,888a6d9d9b2f2738b4f06c53666c86d3,6887669\r\nIXI406,IXI,Guys,58.2834,F,61.8136,IXI406-Guys-0963-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10815/scans/T1/resources/NIfTI/files/IXI406-Guys-0963-T1.nii.gz,7532b840bfc2693840bb2fe70cda3000,7337933\r\nIXI407,IXI,Guys,58.2806,F,61.5131,IXI407-Guys-0964-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10816/scans/T1/resources/NIfTI/files/IXI407-Guys-0964-T1.nii.gz,eef7f336fdcd015b2e72f61a64a36b9d,7601791\r\nIXI408,IXI,Guys,52.1287,F,48.5645,IXI408-Guys-0962-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10817/scans/T1/resources/NIfTI/files/IXI408-Guys-0962-T1.nii.gz,561f87693a7fb4e4275aa519fe560c39,7053659\r\nIXI409,IXI,Guys,70.9514,F,70.2948,IXI409-Guys-0960-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10818/scans/T1/resources/NIfTI/files/IXI409-Guys-0960-T1.nii.gz,ad87bda09ad69b4422728d1894bb62df,7764757\r\nIXI410,IXI,Guys,53.4127,F,47.1459,IXI410-Guys-0958-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10819/scans/T1/resources/NIfTI/files/IXI410-Guys-0958-T1.nii.gz,8df3151c9c21309c708f2e35db6e6866,7964998\r\nIXI411,IXI,Guys,53.4127,F,55.3534,IXI411-Guys-0959-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10820/scans/T1/resources/NIfTI/files/IXI411-Guys-0959-T1.nii.gz,630a048f02165be653ba34008b53aad4,7822817\r\nIXI412,IXI,Guys,58.5681,F,54.508,IXI412-Guys-0949-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10821/scans/T1/resources/NIfTI/files/IXI412-Guys-0949-T1.nii.gz,1f5d4e94ba4995d9e33e8e5bfa7ac4b0,6579334\r\nIXI413,IXI,Guys,59.8357,F,60.6694,IXI413-Guys-0955-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10822/scans/T1/resources/NIfTI/files/IXI413-Guys-0955-T1.nii.gz,13ffd4b16ea9eb72a60a9bcde99d14b8,7005903\r\nIXI414,IXI,Guys,59.7426,F,56.1607,IXI414-Guys-0957-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10823/scans/T1/resources/NIfTI/files/IXI414-Guys-0957-T1.nii.gz,9c48bd8b9680870e749e114c0798e69d,7312621\r\nIXI415,IXI,Guys,72.3258,F,74.4646,IXI415-Guys-0942-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10824/scans/T1/resources/NIfTI/files/IXI415-Guys-0942-T1.nii.gz,0b9117df62453b937e1c5f5b5997d807,7606215\r\nIXI416,IXI,Guys,72.1451,F,70.2053,IXI416-Guys-1051-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10825/scans/T1/resources/NIfTI/files/IXI416-Guys-1051-T1.nii.gz,8b3fb70b3a4b21a536930b2df06dc39c,6724492\r\nIXI417,IXI,Guys,59.8823,M,69.0442,IXI417-Guys-0939-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10826/scans/T1/resources/NIfTI/files/IXI417-Guys-0939-T1.nii.gz,1878aa1e3a05d117103f8c348fb36300,9129499\r\nIXI418,IXI,Guys,59.7426,F,62.6715,IXI418-Guys-0956-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10827/scans/T1/resources/NIfTI/files/IXI418-Guys-0956-T1.nii.gz,b53a88f0aa07f91105478d0737f10705,8056671\r\nIXI419,IXI,Guys,73.1828,F,72.5103,IXI419-Guys-1027-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10828/scans/T1/resources/NIfTI/files/IXI419-Guys-1027-T1.nii.gz,6e9569ece79a8d78441408dbf1a7a01f,7176411\r\nIXI420,IXI,Guys,73.1828,F,73.0233,IXI420-Guys-1028-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10829/scans/T1/resources/NIfTI/files/IXI420-Guys-1028-T1.nii.gz,959fc663334d2f6b637f73a43b005a36,7245181\r\nIXI422,IXI,Guys,37.0979,M,42.4971,IXI422-Guys-1071-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10830/scans/T1/resources/NIfTI/files/IXI422-Guys-1071-T1.nii.gz,bcb10896504c960f5c65ab65d52eac37,8155477\r\nIXI428,IXI,Guys,55.54,F,53.8999,IXI428-Guys-0996-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10836/scans/T1/resources/NIfTI/files/IXI428-Guys-0996-T1.nii.gz,6299ad2e6064654151dde8c7ded86089,7356321\r\nIXI429,IXI,Guys,55.54,F,54.8233,IXI429-Guys-0997-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10837/scans/T1/resources/NIfTI/files/IXI429-Guys-0997-T1.nii.gz,0728b0df40d9698a3cbb09c7161ae4e6,7285206\r\nIXI431,IXI,Guys,58.7817,F,54.5616,IXI431-Guys-0986-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10839/scans/T1/resources/NIfTI/files/IXI431-Guys-0986-T1.nii.gz,dc8b47f613b598c0ccc59f79b43efcb5,6682349\r\nIXI432,IXI,Guys,57.7385,M,66.6781,IXI432-Guys-0987-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10840/scans/T1/resources/NIfTI/files/IXI432-Guys-0987-T1.nii.gz,9b733f3272cdbfcd968cca57a901a77b,8238846\r\nIXI436,IXI,HH,28.3477,M,34.948,IXI436-HH-2153-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10844/scans/T1/resources/NIfTI/files/IXI436-HH-2153-T1.nii.gz,3a1786b84ecf009c5abba92e2fcd3d82,8545402\r\nIXI437,IXI,HH,55.5975,F,54.7131,IXI437-HH-2152-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10845/scans/T1/resources/NIfTI/files/IXI437-HH-2152-T1.nii.gz,954ca2ee0f949ee66ac48512cff0ef8a,8244564\r\nIXI438,IXI,HH,63.6769,M,55.3715,IXI438-HH-2155-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10846/scans/T1/resources/NIfTI/files/IXI438-HH-2155-T1.nii.gz,7e884f9bd3f3b8187cf7c947950e9cb9,8817694\r\nIXI439,IXI,HH,43.9179,F,41.1548,IXI439-HH-2114-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10847/scans/T1/resources/NIfTI/files/IXI439-HH-2114-T1.nii.gz,cffa2e286d89876078a8b3458f0c0ce4,8176781\r\nIXI440,IXI,HH,48.1013,M,49.1366,IXI440-HH-2127-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10848/scans/T1/resources/NIfTI/files/IXI440-HH-2127-T1.nii.gz,b1d163798ef015c53197a85ffa2eb09d,9062735\r\nIXI441,IXI,HH,48.1013,F,43.0871,IXI441-HH-2154-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10849/scans/T1/resources/NIfTI/files/IXI441-HH-2154-T1.nii.gz,6715f632787bb4c92532e58ae6404968,8455954\r\nIXI443,IXI,HH,69.1335,M,72.7346,IXI443-HH-2215-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10851/scans/T1/resources/NIfTI/files/IXI443-HH-2215-T1.nii.gz,82609bf206be7bd6c27a90183eab3e49,9070924\r\nIXI444,IXI,HH,58.3765,F,52.0962,IXI444-HH-2270-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10852/scans/T1/resources/NIfTI/files/IXI444-HH-2270-T1.nii.gz,abe281d874c5fa535334977377ecf07a,8162062\r\nIXI445,IXI,HH,60.0274,M,52.3628,IXI445-HH-2269-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10853/scans/T1/resources/NIfTI/files/IXI445-HH-2269-T1.nii.gz,e04f832148e8ab46174498ed02e5330c,8785148\r\nIXI446,IXI,HH,60.1095,F,53.0891,IXI446-HH-2325-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10854/scans/T1/resources/NIfTI/files/IXI446-HH-2325-T1.nii.gz,9ab8b3da3800d2176953ed6be055edf0,7699646\r\nIXI447,IXI,Guys,72.8268,F,69.792,IXI447-Guys-0979-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10855/scans/T1/resources/NIfTI/files/IXI447-Guys-0979-T1.nii.gz,eec8a746960a0c5390c3095f805918c1,7145184\r\nIXI448,IXI,HH,79.4141,M,76.5813,IXI448-HH-2393-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10856/scans/T1/resources/NIfTI/files/IXI448-HH-2393-T1.nii.gz,2b04d77b9254786a171eb87fa3d25d7f,8809336\r\nIXI449,IXI,Guys,46.8966,M,55.4229,IXI449-Guys-1082-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10857/scans/T1/resources/NIfTI/files/IXI449-Guys-1082-T1.nii.gz,85162a43612c7cfdda0895e241caf907,8399572\r\nIXI450,IXI,Guys,68.5092,M,73.4489,IXI450-Guys-1093-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10858/scans/T1/resources/NIfTI/files/IXI450-Guys-1093-T1.nii.gz,7c2362aeef2899acff34d67db332cebe,7438732\r\nIXI451,IXI,HH,41.1307,F,33.6968,IXI451-HH-2212-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10859/scans/T1/resources/NIfTI/files/IXI451-HH-2212-T1.nii.gz,b2c50eb2da772377a58544856e4d7d4d,8105743\r\nIXI452,IXI,HH,63.4031,F,68.9099,IXI452-HH-2213-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10860/scans/T1/resources/NIfTI/files/IXI452-HH-2213-T1.nii.gz,576905edf4bff80e2ec2b2e38a6f5651,8350438\r\nIXI453,IXI,HH,63.4168,M,70.1273,IXI453-HH-2214-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10861/scans/T1/resources/NIfTI/files/IXI453-HH-2214-T1.nii.gz,33f10eacb3df20034ca1a4c61d77b13d,9292573\r\nIXI454,IXI,Guys,32.5777,M,39.9302,IXI454-Guys-1083-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10862/scans/T1/resources/NIfTI/files/IXI454-Guys-1083-T1.nii.gz,10b8a14c572cb3eb4201e5e670035888,8626695\r\nIXI455,IXI,Guys,68.4819,F,72.3523,IXI455-Guys-0981-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10863/scans/T1/resources/NIfTI/files/IXI455-Guys-0981-T1.nii.gz,e37419461029917120d27c2f0fa9fd6a,7913616\r\nIXI456,IXI,Guys,70.2478,F,64.0325,IXI456-Guys-1019-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10864/scans/T1/resources/NIfTI/files/IXI456-Guys-1019-T1.nii.gz,8d7ad300786775cda3e0fc05c52c60dd,7380803\r\nIXI458,IXI,Guys,70.7105,M,68.0548,IXI458-Guys-0993-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10866/scans/T1/resources/NIfTI/files/IXI458-Guys-0993-T1.nii.gz,ca91f4032be6ea6fa6f340482016d391,8271030\r\nIXI459,IXI,Guys,62.4257,F,53.1426,IXI459-Guys-0992-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10867/scans/T1/resources/NIfTI/files/IXI459-Guys-0992-T1.nii.gz,047873f07b2711343a0cd852a0709603,7015676\r\nIXI460,IXI,Guys,60.8953,F,58.5401,IXI460-Guys-0999-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10868/scans/T1/resources/NIfTI/files/IXI460-Guys-0999-T1.nii.gz,5619946cafabf9ec47a8b996e55b7c42,8089650\r\nIXI461,IXI,Guys,60.8953,F,58.021,IXI461-Guys-0998-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10869/scans/T1/resources/NIfTI/files/IXI461-Guys-0998-T1.nii.gz,f3d8d4a2e9b3a505e13033a54eeeefa7,7778813\r\nIXI465,IXI,HH,69.2813,M,73.4162,IXI465-HH-2176-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10873/scans/T1/resources/NIfTI/files/IXI465-HH-2176-T1.nii.gz,1ce74a089ae461d306619827dd7f2287,8064128\r\nIXI467,IXI,Guys,66.9487,F,69.8731,IXI467-Guys-0983-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10874/scans/T1/resources/NIfTI/files/IXI467-Guys-0983-T1.nii.gz,ba9c7c2bd82693c3176610902e34e201,7064088\r\nIXI468,IXI,Guys,67.6988,M,61.9101,IXI468-Guys-0985-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10875/scans/T1/resources/NIfTI/files/IXI468-Guys-0985-T1.nii.gz,4037015fac7e84242aba5307adb1693d,8367985\r\nIXI479,IXI,Guys,60.0164,F,58.7789,IXI479-Guys-1034-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10884/scans/T1/resources/NIfTI/files/IXI479-Guys-1034-T1.nii.gz,72b473fe7029b44dacd739c90fcef1fe,6995125\r\nIXI480,IXI,Guys,60.0164,F,55.0565,IXI480-Guys-1033-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10885/scans/T1/resources/NIfTI/files/IXI480-Guys-1033-T1.nii.gz,4c79b6355baeeba6711cd3354e0add6e,7118080\r\nIXI481,IXI,HH,65.3306,M,61.5513,IXI481-HH-2175-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10886/scans/T1/resources/NIfTI/files/IXI481-HH-2175-T1.nii.gz,36b7822de4cb79cd19f69b70b6d6e43d,9692765\r\nIXI482,IXI,HH,72.0903,F,67.3153,IXI482-HH-2178-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10887/scans/T1/resources/NIfTI/files/IXI482-HH-2178-T1.nii.gz,ebf4c1bf9613d5263f99d892715d39cf,8539599\r\nIXI483,IXI,HH,63.3922,F,58.4932,IXI483-HH-2177-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10888/scans/T1/resources/NIfTI/files/IXI483-HH-2177-T1.nii.gz,fe7f7211ea7dd577ab0c6bc0bc7e7526,8630202\r\nIXI484,IXI,HH,66.3737,M,61.3249,IXI484-HH-2179-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10889/scans/T1/resources/NIfTI/files/IXI484-HH-2179-T1.nii.gz,a2e881e0754cc3b7b9b447956f17b3f2,9015413\r\nIXI485,IXI,HH,68.8515,F,63.6147,IXI485-HH-2180-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10890/scans/T1/resources/NIfTI/files/IXI485-HH-2180-T1.nii.gz,1d9503b319d53a49090d326aa6381728,8779040\r\nIXI486,IXI,Guys,63.8357,F,60.0647,IXI486-Guys-1005-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10891/scans/T1/resources/NIfTI/files/IXI486-Guys-1005-T1.nii.gz,86e7f51e643cfcfab057f203039c59ff,6996987\r\nIXI487,IXI,Guys,66.2752,M,58.147,IXI487-Guys-1037-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10892/scans/T1/resources/NIfTI/files/IXI487-Guys-1037-T1.nii.gz,006972754ea3e0ffb511be0148bd3cb5,8997070\r\nIXI488,IXI,Guys,69.0678,F,73.3948,IXI488-Guys-1015-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10893/scans/T1/resources/NIfTI/files/IXI488-Guys-1015-T1.nii.gz,04c6cfed25baad615c846478aec2597a,7036791\r\nIXI489,IXI,Guys,69.0678,F,68.4137,IXI489-Guys-1014-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10894/scans/T1/resources/NIfTI/files/IXI489-Guys-1014-T1.nii.gz,5cef6c434eaebeb3745ffba8f7dcb641,6696722\r\nIXI490,IXI,Guys,70.141,F,65.7168,IXI490-Guys-1018-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10895/scans/T1/resources/NIfTI/files/IXI490-Guys-1018-T1.nii.gz,9048e277e6fdd5ef94f91956552175de,8032145\r\nIXI491,IXI,Guys,75.9726,F,63.8424,IXI491-Guys-1032-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10896/scans/T1/resources/NIfTI/files/IXI491-Guys-1032-T1.nii.gz,6f6c70f761fd20b6138f052ba26e8ccf,7613366\r\nIXI492,IXI,Guys,69.2895,F,67.5012,IXI492-Guys-1022-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10897/scans/T1/resources/NIfTI/files/IXI492-Guys-1022-T1.nii.gz,70d9b2e4244c3110cae3d254dfcd5f86,7754402\r\nIXI493,IXI,Guys,67.7673,M,61.3481,IXI493-Guys-1007-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10898/scans/T1/resources/NIfTI/files/IXI493-Guys-1007-T1.nii.gz,648c4276969e5a2ba80028ecc5e806c3,7872662\r\nIXI494,IXI,Guys,66.3326,M,68.762,IXI494-Guys-1008-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10899/scans/T1/resources/NIfTI/files/IXI494-Guys-1008-T1.nii.gz,81e6450832f79c947f5ccfb87e84480f,7833896\r\nIXI495,IXI,Guys,60.8679,F,62.9528,IXI495-Guys-1009-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10900/scans/T1/resources/NIfTI/files/IXI495-Guys-1009-T1.nii.gz,ad407b90aa052425bb95ca92857dccb5,7521100\r\nIXI496,IXI,Guys,62.8036,F,64.9471,IXI496-Guys-1045-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10901/scans/T1/resources/NIfTI/files/IXI496-Guys-1045-T1.nii.gz,b7276109222f816d3c0bcfdf99a61155,6965659\r\nIXI497,IXI,Guys,72.3066,M,76.1273,IXI497-Guys-1002-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10902/scans/T1/resources/NIfTI/files/IXI497-Guys-1002-T1.nii.gz,215f2e4ee7a7f380484e9e5033eeae9c,9046208\r\nIXI498,IXI,Guys,76.7803,F,69.6752,IXI498-Guys-1050-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10903/scans/T1/resources/NIfTI/files/IXI498-Guys-1050-T1.nii.gz,dbd48222b5fec10b38363d276014236b,7759796\r\nIXI499,IXI,Guys,82.1875,M,76.9275,IXI499-Guys-1004-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10904/scans/T1/resources/NIfTI/files/IXI499-Guys-1004-T1.nii.gz,13ce03ab974f30b1a7882926388fa027,7673238\r\nIXI500,IXI,Guys,63.1814,M,58.0117,IXI500-Guys-1017-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10905/scans/T1/resources/NIfTI/files/IXI500-Guys-1017-T1.nii.gz,63dc0d44f00b07b99fa7566bb1bc41b2,8055688\r\nIXI501,IXI,Guys,67.1129,F,67.7471,IXI501-Guys-1016-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10906/scans/T1/resources/NIfTI/files/IXI501-Guys-1016-T1.nii.gz,7d012caa94f7c07064d4cdc81c3655fc,7536863\r\nIXI502,IXI,Guys,63.8768,F,53.1496,IXI502-Guys-1020-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10907/scans/T1/resources/NIfTI/files/IXI502-Guys-1020-T1.nii.gz,a41ae78d127fc9485a8fb46242b4f497,7194961\r\nIXI503,IXI,Guys,65.0951,M,73.084,IXI503-Guys-1021-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10908/scans/T1/resources/NIfTI/files/IXI503-Guys-1021-T1.nii.gz,c419d641f0c10e3f460d3fbe89fd69c9,8208722\r\nIXI504,IXI,Guys,68.1506,M,67.1594,IXI504-Guys-1025-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10909/scans/T1/resources/NIfTI/files/IXI504-Guys-1025-T1.nii.gz,8f3869bf2d165e091af2768ed66596f8,9419991\r\nIXI505,IXI,Guys,62.245,F,65.834,IXI505-Guys-1026-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10910/scans/T1/resources/NIfTI/files/IXI505-Guys-1026-T1.nii.gz,ecb9b592492bb0ff8fc8bf9c959071b4,7280601\r\nIXI506,IXI,Guys,69.6893,F,71.6943,IXI506-Guys-1035-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10911/scans/T1/resources/NIfTI/files/IXI506-Guys-1035-T1.nii.gz,c9063acd68b9d26c1c76409570460de7,7566927\r\nIXI507,IXI,Guys,71.6632,M,70.5657,IXI507-Guys-1036-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10912/scans/T1/resources/NIfTI/files/IXI507-Guys-1036-T1.nii.gz,e53d803f15e367e668472b807aa5fd2b,7967066\r\nIXI508,IXI,HH,60.9309,F,60.3613,IXI508-HH-2268-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10913/scans/T1/resources/NIfTI/files/IXI508-HH-2268-T1.nii.gz,bffc4862df65ce3040049c26fba53267,8118260\r\nIXI511,IXI,HH,64.5777,F,66.9467,IXI511-HH-2238-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10915/scans/T1/resources/NIfTI/files/IXI511-HH-2238-T1.nii.gz,d5a5dfc07445c753596a3a93a5047ba5,7739902\r\nIXI512,IXI,Guys,64.7502,M,66.7776,IXI512-Guys-1054-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10916/scans/T1/resources/NIfTI/files/IXI512-Guys-1054-T1.nii.gz,58c9f67ef60b0daa808a3f5b2ffc245d,8354864\r\nIXI515,IXI,HH,62.089,F,55.4974,IXI515-HH-2377-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10917/scans/T1/resources/NIfTI/files/IXI515-HH-2377-T1.nii.gz,828dcabdcde0b7384c471fb4c47bad52,9250368\r\nIXI516,IXI,HH,60.4107,F,64.7989,IXI516-HH-2297-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10918/scans/T1/resources/NIfTI/files/IXI516-HH-2297-T1.nii.gz,d516ae5f7521fa28ee36180f8ec65b5e,8470376\r\nIXI518,IXI,HH,75.8275,F,70.905,IXI518-HH-2239-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10920/scans/T1/resources/NIfTI/files/IXI518-HH-2239-T1.nii.gz,f75485be1ae44943a5424c49c1732ad6,8370164\r\nIXI519,IXI,HH,57.5058,M,53.0319,IXI519-HH-2240-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10921/scans/T1/resources/NIfTI/files/IXI519-HH-2240-T1.nii.gz,ade01a723eb72a312a908981286d4bea,8210571\r\nIXI521,IXI,HH,70.8364,F,70.1715,IXI521-HH-2353-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10922/scans/T1/resources/NIfTI/files/IXI521-HH-2353-T1.nii.gz,b7cf4412624ec0873d1804ddc115a05e,7594241\r\nIXI522,IXI,HH,28.9336,F,33.1666,IXI522-HH-2453-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10923/scans/T1/resources/NIfTI/files/IXI522-HH-2453-T1.nii.gz,5515ab692fad98383623d3b6be063f2f,8038980\r\nIXI523,IXI,Guys,63.4196,F,68.3919,IXI523-Guys-1056-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10924/scans/T1/resources/NIfTI/files/IXI523-Guys-1056-T1.nii.gz,8704e751f67e290ff1a8f098be0a3050,6430833\r\nIXI524,IXI,HH,33.5168,F,42.783,IXI524-HH-2412-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10925/scans/T1/resources/NIfTI/files/IXI524-HH-2412-T1.nii.gz,24a81ebc9f872ad847f09988bd80686f,8656413\r\nIXI525,IXI,HH,41.5195,M,45.0045,IXI525-HH-2413-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10926/scans/T1/resources/NIfTI/files/IXI525-HH-2413-T1.nii.gz,33e23eca820455eeb6b5ca9bae1c269a,9439728\r\nIXI526,IXI,HH,54.6092,F,52.8661,IXI526-HH-2392-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10927/scans/T1/resources/NIfTI/files/IXI526-HH-2392-T1.nii.gz,6cab6d30a8517c914c4fed3ae81a59d8,8979446\r\nIXI528,IXI,Guys,40.5284,M,41.6842,IXI528-Guys-1073-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10929/scans/T1/resources/NIfTI/files/IXI528-Guys-1073-T1.nii.gz,f972aaf57b41d6cf60b91c54b953d50f,8760944\r\nIXI531,IXI,Guys,75.937,M,70.3937,IXI531-Guys-1057-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10930/scans/T1/resources/NIfTI/files/IXI531-Guys-1057-T1.nii.gz,b7f63f5137657f1b958424c013a75803,8229484\r\nIXI533,IXI,Guys,73.6044,M,71.3457,IXI533-Guys-1066-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10932/scans/T1/resources/NIfTI/files/IXI533-Guys-1066-T1.nii.gz,83fa782f51c1ae1ba4ee8bde0134e8e2,3925450\r\nIXI534,IXI,Guys,69.6099,M,65.8548,IXI534-Guys-1062-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10933/scans/T1/resources/NIfTI/files/IXI534-Guys-1062-T1.nii.gz,9302b36cb664e1c40ebd961041f03faa,8974270\r\nIXI535,IXI,Guys,65.2594,F,68.3796,IXI535-Guys-1061-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10934/scans/T1/resources/NIfTI/files/IXI535-Guys-1061-T1.nii.gz,f47fd018cf107528d7f69fa8bf01a624,7511530\r\nIXI536,IXI,Guys,58.8309,F,67.2056,IXI536-Guys-1059-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10935/scans/T1/resources/NIfTI/files/IXI536-Guys-1059-T1.nii.gz,93139528e521cfdee9a36dad702ea6f7,7152711\r\nIXI537,IXI,HH,67.4552,F,68.3573,IXI537-HH-2378-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10936/scans/T1/resources/NIfTI/files/IXI537-HH-2378-T1.nii.gz,a90f7e3d6b8dd858cba1fd7983d78127,9081330\r\nIXI538,IXI,HH,81.9411,F,77.6075,IXI538-HH-2411-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10937/scans/T1/resources/NIfTI/files/IXI538-HH-2411-T1.nii.gz,e22922e8cbff1a90d6f733f1526d61db,8432820\r\nIXI539,IXI,Guys,78.0726,M,74.1365,IXI539-Guys-1067-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10938/scans/T1/resources/NIfTI/files/IXI539-Guys-1067-T1.nii.gz,93ad4a072c4d300ae90b2cc034abdd9b,8048116\r\nIXI544,IXI,HH,58.0424,F,64.7217,IXI544-HH-2395-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10942/scans/T1/resources/NIfTI/files/IXI544-HH-2395-T1.nii.gz,5dcab13ad078335495142909122bbd71,8566212\r\nIXI546,IXI,HH,60.0739,F,57.375,IXI546-HH-2450-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10943/scans/T1/resources/NIfTI/files/IXI546-HH-2450-T1.nii.gz,a9d3984c6c6e1f3e2280c2a438b7b0da,8168270\r\nIXI549,IXI,Guys,25.0185,F,21.1461,IXI549-Guys-1046-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10946/scans/T1/resources/NIfTI/files/IXI549-Guys-1046-T1.nii.gz,75409a291bfa6614abbd06dad215a136,7555537\r\nIXI550,IXI,Guys,59.9069,F,55.9791,IXI550-Guys-1069-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10947/scans/T1/resources/NIfTI/files/IXI550-Guys-1069-T1.nii.gz,1bd45bcba06c6aed55470898c42dc4ca,7506792\r\nIXI551,IXI,Guys,70.5024,F,70.132,IXI551-Guys-1065-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10948/scans/T1/resources/NIfTI/files/IXI551-Guys-1065-T1.nii.gz,df66dc15795b48f929a2b2c0319e0224,7391167\r\nIXI552,IXI,Guys,64.6626,F,63.3486,IXI552-Guys-1063-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10949/scans/T1/resources/NIfTI/files/IXI552-Guys-1063-T1.nii.gz,2b9716806688f67316498113609b3fec,7626428\r\nIXI554,IXI,Guys,70.1054,M,72.3303,IXI554-Guys-1068-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10951/scans/T1/resources/NIfTI/files/IXI554-Guys-1068-T1.nii.gz,c68a2a990f23dcff1243bd1584a7d0a7,8613310\r\nIXI555,IXI,Guys,65.6674,F,67.494,IXI555-Guys-1074-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10952/scans/T1/resources/NIfTI/files/IXI555-Guys-1074-T1.nii.gz,5c9fdb255525138c098b0164c01fd4f6,8471991\r\nIXI556,IXI,HH,55.6359,M,60.6518,IXI556-HH-2452-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10953/scans/T1/resources/NIfTI/files/IXI556-HH-2452-T1.nii.gz,9b7508764feecb5091e10b5138c33dcc,9405618\r\nIXI558,IXI,Guys,66.8583,F,65.229,IXI558-Guys-1079-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10954/scans/T1/resources/NIfTI/files/IXI558-Guys-1079-T1.nii.gz,230a12c7f7f7ec1c538ceac20b90a15e,7294644\r\nIXI559,IXI,HH,67.8193,F,62.9873,IXI559-HH-2394-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10955/scans/T1/resources/NIfTI/files/IXI559-HH-2394-T1.nii.gz,5b6b9bb5fca10bcb747a0e7eb839e805,8271130\r\nIXI560,IXI,Guys,63.8686,M,61.3302,IXI560-Guys-1070-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10956/scans/T1/resources/NIfTI/files/IXI560-Guys-1070-T1.nii.gz,9b0b249a7d0d44cce9af26a8ea8085b0,8250041\r\nIXI562,IXI,Guys,42.7899,F,42.1459,IXI562-Guys-1131-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10958/scans/T1/resources/NIfTI/files/IXI562-Guys-1131-T1.nii.gz,4b36422355647fe4ccf341261d0ca7e0,7217359\r\nIXI565,IXI,HH,63.8166,M,65.6832,IXI565-HH-2534-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10960/scans/T1/resources/NIfTI/files/IXI565-HH-2534-T1.nii.gz,8151eff0eaf014289ae86fb52e444a2d,9534145\r\nIXI566,IXI,HH,42.9706,F,44.888,IXI566-HH-2535-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10961/scans/T1/resources/NIfTI/files/IXI566-HH-2535-T1.nii.gz,cc72f349eaac7db9ed58f82017b3af0c,8965213\r\nIXI567,IXI,HH,28.5613,F,36.2961,IXI567-HH-2536-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10962/scans/T1/resources/NIfTI/files/IXI567-HH-2536-T1.nii.gz,e3b0aa4b6252025039a991b39841d4c9,7501682\r\nIXI568,IXI,HH,33.6865,F,40.5223,IXI568-HH-2607-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10963/scans/T1/resources/NIfTI/files/IXI568-HH-2607-T1.nii.gz,eb549f9b61f1d75efed0d75f5e0ef723,8209288\r\nIXI569,IXI,Guys,56.9637,M,49.8317,IXI569-Guys-1101-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10964/scans/T1/resources/NIfTI/files/IXI569-Guys-1101-T1.nii.gz,282fc5e6375cb771647157e695084ecf,7903792\r\nIXI572,IXI,HH,41.9493,F,33.3804,IXI572-HH-2605-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10966/scans/T1/resources/NIfTI/files/IXI572-HH-2605-T1.nii.gz,b8466021aea23282c821bcfb76cc471a,8058530\r\nIXI575,IXI,HH,57.755,F,52.1881,IXI575-HH-2658-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10969/scans/T1/resources/NIfTI/files/IXI575-HH-2658-T1.nii.gz,c0c8674f84c3edab4a1fda50a808df59,9048747\r\nIXI576,IXI,Guys,67.332,F,67.8172,IXI576-Guys-1077-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10970/scans/T1/resources/NIfTI/files/IXI576-Guys-1077-T1.nii.gz,da78ed15ffe95effdfede3fa6820ebc2,7297369\r\nIXI577,IXI,HH,64.1862,F,61.1415,IXI577-HH-2661-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10971/scans/T1/resources/NIfTI/files/IXI577-HH-2661-T1.nii.gz,035a3c296718d5772c374aa2ae0d6dfe,8111661\r\nIXI578,IXI,Guys,57.8645,M,63.6243,IXI578-Guys-1078-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10972/scans/T1/resources/NIfTI/files/IXI578-Guys-1078-T1.nii.gz,13a53d1952eb3e196ac13c410c521677,9018678\r\nIXI579,IXI,Guys,41.2156,M,47.3319,IXI579-Guys-1126-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10973/scans/T1/resources/NIfTI/files/IXI579-Guys-1126-T1.nii.gz,2e4ce5f058322fa3e0e59e6747d58d7e,8361171\r\nIXI582,IXI,Guys,29.7714,M,40.0257,IXI582-Guys-1127-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10975/scans/T1/resources/NIfTI/files/IXI582-Guys-1127-T1.nii.gz,d77aace6e94eb4060088ec44827d8020,8225627\r\nIXI584,IXI,Guys,41.3279,M,50.1704,IXI584-Guys-1129-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10976/scans/T1/resources/NIfTI/files/IXI584-Guys-1129-T1.nii.gz,fc948f73b54864ef3c2ab41b11c65eae,8959340\r\nIXI585,IXI,Guys,28.1205,M,40.1783,IXI585-Guys-1130-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10977/scans/T1/resources/NIfTI/files/IXI585-Guys-1130-T1.nii.gz,afd6cb9c1f10922e38a46a970f9e6a6d,8121458\r\nIXI586,IXI,HH,34.3655,M,37.197,IXI586-HH-2451-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10978/scans/T1/resources/NIfTI/files/IXI586-HH-2451-T1.nii.gz,0ab7721e38874a595534e839ff2aa87d,8680220\r\nIXI587,IXI,Guys,33.5934,M,41.1488,IXI587-Guys-1128-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10979/scans/T1/resources/NIfTI/files/IXI587-Guys-1128-T1.nii.gz,7468d7bd484dc5b1ef939db19ec02bc8,8730346\r\nIXI591,IXI,Guys,59.8932,F,63.2961,IXI591-Guys-1084-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10982/scans/T1/resources/NIfTI/files/IXI591-Guys-1084-T1.nii.gz,22c9311bf0b0b913c493d4e623d4c25e,7018352\r\nIXI592,IXI,Guys,48.0794,M,49.7364,IXI592-Guys-1085-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10983/scans/T1/resources/NIfTI/files/IXI592-Guys-1085-T1.nii.gz,61117edbe987404243cf4158e55bfb19,7501154\r\nIXI593,IXI,Guys,55.2197,F,66.2707,IXI593-Guys-1109-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10984/scans/T1/resources/NIfTI/files/IXI593-Guys-1109-T1.nii.gz,26a3f60633de48abfbd8ed5b8225054d,11465304\r\nIXI594,IXI,Guys,61.9959,M,63.4764,IXI594-Guys-1089-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10985/scans/T1/resources/NIfTI/files/IXI594-Guys-1089-T1.nii.gz,9085c839c97c6de76b5cfcbe941b5573,8871887\r\nIXI598,IXI,HH,52.8871,M,53.8556,IXI598-HH-2606-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10989/scans/T1/resources/NIfTI/files/IXI598-HH-2606-T1.nii.gz,5b34f233eb4ef24a6b7acd73ea089825,9315597\r\nIXI599,IXI,HH,39.384,M,42.821,IXI599-HH-2659-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10990/scans/T1/resources/NIfTI/files/IXI599-HH-2659-T1.nii.gz,240a61d6f67933867060b911fb64a69d,9285390\r\nIXI600,IXI,HH,38.9596,M,32.001,IXI600-HH-2660-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10991/scans/T1/resources/NIfTI/files/IXI600-HH-2660-T1.nii.gz,535feb2ae4dc7feaa48fc04e24f4124a,8851260\r\nIXI601,IXI,HH,35.5154,M,31.9265,IXI601-HH-2700-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10992/scans/T1/resources/NIfTI/files/IXI601-HH-2700-T1.nii.gz,6b2c7901526e615043fa088cd8afb8c3,8370967\r\nIXI603,IXI,HH,66.0151,F,61.4281,IXI603-HH-2701-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10993/scans/T1/resources/NIfTI/files/IXI603-HH-2701-T1.nii.gz,7bf2c8743966f908d740f79a54d1fc76,8836757\r\nIXI605,IXI,HH,70.3326,F,71.7589,IXI605-HH-2598-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10994/scans/T1/resources/NIfTI/files/IXI605-HH-2598-T1.nii.gz,d4e676d199c39604a97c952d332a4e20,8694601\r\nIXI606,IXI,HH,60.5394,M,63.3381,IXI606-HH-2601-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10995/scans/T1/resources/NIfTI/files/IXI606-HH-2601-T1.nii.gz,2bc49df570c534f90b61416b657fb83b,9153433\r\nIXI607,IXI,Guys,83.8111,M,77.3927,IXI607-Guys-1097-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10996/scans/T1/resources/NIfTI/files/IXI607-Guys-1097-T1.nii.gz,21e1e79a86a9cdb236f7e2fbd1efceb9,8250169\r\nIXI608,IXI,HH,52.4983,M,55.9486,IXI608-HH-2599-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10997/scans/T1/resources/NIfTI/files/IXI608-HH-2599-T1.nii.gz,593bdce1844d8334d580086549643ced,8989078\r\nIXI609,IXI,HH,28.9966,M,36.4199,IXI609-HH-2600-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10998/scans/T1/resources/NIfTI/files/IXI609-HH-2600-T1.nii.gz,bd989b934860cf0ec4c569a426583c1e,7963768\r\nIXI610,IXI,HH,56.6762,M,45.4491,IXI610-HH-2649-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E10999/scans/T1/resources/NIfTI/files/IXI610-HH-2649-T1.nii.gz,c44f233ae875e98828498ad0a5528977,8338338\r\nIXI611,IXI,HH,28.8706,M,34.2305,IXI611-HH-2650-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11000/scans/T1/resources/NIfTI/files/IXI611-HH-2650-T1.nii.gz,5a38f479f0616f920681f8091af11b63,8813093\r\nIXI612,IXI,HH,33.9165,M,35.7291,IXI612-HH-2688-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11001/scans/T1/resources/NIfTI/files/IXI612-HH-2688-T1.nii.gz,591faebead51b4ea0914e4a32977f20d,8557577\r\nIXI613,IXI,HH,25.5852,M,26.301,IXI613-HH-2734-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11002/scans/T1/resources/NIfTI/files/IXI613-HH-2734-T1.nii.gz,8b1943135b4a8bc22bfa8d0fb8a3b217,9053790\r\nIXI614,IXI,HH,26.9295,F,36.9083,IXI614-HH-2735-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11003/scans/T1/resources/NIfTI/files/IXI614-HH-2735-T1.nii.gz,eee10e9096b54336f983b5dc2bbdb2eb,7941511\r\nIXI616,IXI,Guys,55.0883,M,59.1932,IXI616-Guys-1092-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11004/scans/T1/resources/NIfTI/files/IXI616-Guys-1092-T1.nii.gz,6781b00bd129d0ad1f6ffe59dbf235af,8308683\r\nIXI617,IXI,Guys,39.1759,M,48.5407,IXI617-Guys-1090-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11005/scans/T1/resources/NIfTI/files/IXI617-Guys-1090-T1.nii.gz,3bea9c2071bd15994226b67ba892834c,8271796\r\nIXI618,IXI,Guys,37.9493,M,46.8922,IXI618-Guys-1091-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11006/scans/T1/resources/NIfTI/files/IXI618-Guys-1091-T1.nii.gz,183bd964bd2758e37776014a301b5034,8045348\r\nIXI619,IXI,Guys,24.8761,M,31.8305,IXI619-Guys-1099-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11007/scans/T1/resources/NIfTI/files/IXI619-Guys-1099-T1.nii.gz,310fe65821509a631577b91f07ef0d97,7919784\r\nIXI621,IXI,Guys,31.6167,M,41.9405,IXI621-Guys-1100-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11008/scans/T1/resources/NIfTI/files/IXI621-Guys-1100-T1.nii.gz,5bca0850ff7651affa95a17a11e63cef,7301911\r\nIXI622,IXI,Guys,59.6851,M,58.0399,IXI622-Guys-1102-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11009/scans/T1/resources/NIfTI/files/IXI622-Guys-1102-T1.nii.gz,6f553238dd98eb9bfde5bd911a4c664d,8462359\r\nIXI625,IXI,Guys,47.0691,M,45.4128,IXI625-Guys-1098-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11011/scans/T1/resources/NIfTI/files/IXI625-Guys-1098-T1.nii.gz,02057cafec75ef57d4eec8d201483029,8148154\r\nIXI626,IXI,Guys,64.2683,M,59.7209,IXI626-Guys-1094-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11012/scans/T1/resources/NIfTI/files/IXI626-Guys-1094-T1.nii.gz,bae49c5c08ec5cdb689dfa674054dc66,8479672\r\nIXI627,IXI,Guys,68.4846,M,65.5053,IXI627-Guys-1103-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11013/scans/T1/resources/NIfTI/files/IXI627-Guys-1103-T1.nii.gz,78bafb0e4ceb2375d8594b94e9d0e6c6,8046479\r\nIXI629,IXI,Guys,59.2635,M,62.2011,IXI629-Guys-1095-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11014/scans/T1/resources/NIfTI/files/IXI629-Guys-1095-T1.nii.gz,6a5f032fe083ef18711aba0e54d5a648,8886309\r\nIXI630,IXI,Guys,54.6858,M,62.1331,IXI630-Guys-1108-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11015/scans/T1/resources/NIfTI/files/IXI630-Guys-1108-T1.nii.gz,30d807aa8c6f35eea65f84dd00e9413d,4878652\r\nIXI631,IXI,HH,41.3005,M,43.5481,IXI631-HH-2651-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11016/scans/T1/resources/NIfTI/files/IXI631-HH-2651-T1.nii.gz,7f1f728f16a18b78b5070238f6655e00,8727162\r\nIXI632,IXI,HH,56.4381,M,61.4827,IXI632-HH-2652-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11017/scans/T1/resources/NIfTI/files/IXI632-HH-2652-T1.nii.gz,72de67059ebca7926866564a4e0629cb,8813871\r\nIXI633,IXI,HH,36.7064,M,40.4993,IXI633-HH-2689-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11018/scans/T1/resources/NIfTI/files/IXI633-HH-2689-T1.nii.gz,9553956703dc92ff293483b5e342f80e,8120226\r\nIXI634,IXI,HH,34.0561,M,42.2475,IXI634-HH-2690-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11019/scans/T1/resources/NIfTI/files/IXI634-HH-2690-T1.nii.gz,5d5fa9b9752f3a5ed9a07a317bd9e6ac,8581049\r\nIXI635,IXI,HH,25.5305,M,27.7846,IXI635-HH-2691-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11020/scans/T1/resources/NIfTI/files/IXI635-HH-2691-T1.nii.gz,d807a3d22ff928c25735409f3d574018,8575354\r\nIXI636,IXI,HH,43.1567,M,51.2745,IXI636-HH-2733-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11021/scans/T1/resources/NIfTI/files/IXI636-HH-2733-T1.nii.gz,6a325605ad3a47fe6f936a9123ca6e7f,9503118\r\nIXI639,IXI,Guys,86.1985,M,75.6134,IXI639-Guys-1088-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11024/scans/T1/resources/NIfTI/files/IXI639-Guys-1088-T1.nii.gz,7bdc259fe7a4da1bc341187e34636da2,8460185\r\nIXI640,IXI,Guys,80.8022,M,78.7217,IXI640-Guys-1106-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11025/scans/T1/resources/NIfTI/files/IXI640-Guys-1106-T1.nii.gz,ff49ae308a389149aeb5aaa6060ab183,8609988\r\nIXI641,IXI,Guys,68.1232,M,68.4176,IXI641-Guys-1105-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11026/scans/T1/resources/NIfTI/files/IXI641-Guys-1105-T1.nii.gz,f3a8e6020129d2b459a14fb6d9064082,8766843\r\nIXI642,IXI,Guys,57.0678,M,56.3124,IXI642-Guys-1104-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11027/scans/T1/resources/NIfTI/files/IXI642-Guys-1104-T1.nii.gz,0f573b28821f7789fc91191e93ebbc70,8639634\r\nIXI644,IXI,Guys,56.8734,M,47.1797,IXI644-Guys-1121-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11029/scans/T1/resources/NIfTI/files/IXI644-Guys-1121-T1.nii.gz,e9ba6a26390fc4e2cb8837b1c4c000c2,9532005\r\nIXI646,IXI,HH,71.206,F,74.5827,IXI646-HH-2653-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11030/scans/T1/resources/NIfTI/files/IXI646-HH-2653-T1.nii.gz,2b53de3a7c2ceddb3379069c6b485eb0,8664940\r\nIXI648,IXI,Guys,47.7235,M,52.5089,IXI648-Guys-1107-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11031/scans/T1/resources/NIfTI/files/IXI648-Guys-1107-T1.nii.gz,6656b1516328d5f4339e7efadc4a93be,8163431\r\nIXI651,IXI,Guys,50.3956,M,56.4696,IXI651-Guys-1118-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11032/scans/T1/resources/NIfTI/files/IXI651-Guys-1118-T1.nii.gz,c5aa43e672bca73e86e7014ccc912c6e,7584957\r\nIXI652,IXI,Guys,42.9897,M,45.0112,IXI652-Guys-1116-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11033/scans/T1/resources/NIfTI/files/IXI652-Guys-1116-T1.nii.gz,2711e91f24eace035757c6a48f38fe5b,9243260\r\nIXI653,IXI,Guys,46.2204,M,46.1552,IXI653-Guys-1122-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11034/scans/T1/resources/NIfTI/files/IXI653-Guys-1122-T1.nii.gz,8d0ad54496d79755f60b60ceb55c6068,8572370\r\nIXI662,IXI,Guys,41.7413,M,56.4527,IXI662-Guys-1120-T1.nii.gz,https://www.nitrc.org/ir/data/experiments/NITRC_IR_E11038/scans/T1/resources/NIfTI/files/IXI662-Guys-1120-T1.nii.gz,0eb444a6145bf9f9cf94d2ffb82deaa1,8811031\r\neNKI_A00028552,eNKI,eNKI,61.0,F,67.9876,sub-A00028552_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00028552/ses-DS2/anat/sub-A00028552_ses-DS2_T1w.nii.gz,0eecaab0f85590fab5b725534033455b,5173311\r\neNKI_A00030947,eNKI,eNKI,36.0,F,53.293,sub-A00030947_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00030947/ses-ALGA/ses-ALGA/anat/sub-A00030947_ses-ALGA_T1w.nii.gz,c9c53433bd5dcf74c20aac668299cdab,6326353\r\neNKI_A00033247,eNKI,eNKI,22.0,M,31.3202,sub-A00033247_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00033247/ses-DS2/anat/sub-A00033247_ses-DS2_T1w.nii.gz,7fa17e2112595a8d1467fc60fea18958,5730869\r\neNKI_A00034854,eNKI,eNKI,21.0,F,29.5008,sub-A00034854_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00034854/ses-DS2/anat/sub-A00034854_ses-DS2_T1w.nii.gz,8c026815607b0d4daa2368c2d594606d,5464180\r\neNKI_A00035378,eNKI,eNKI,54.0,F,52.3543,sub-A00035378_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00035378/ses-DS2/anat/sub-A00035378_ses-DS2_T1w.nii.gz,979e0e1507502edc44d47322e99957ef,4245029\r\neNKI_A00037267,eNKI,eNKI,54.0,F,53.5984,sub-A00037267_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00037267/ses-ALGA/ses-ALGA/anat/sub-A00037267_ses-ALGA_T1w.nii.gz,6dd1b4cacfff352c00b8c2186a09a37f,7546488\r\neNKI_A00037386,eNKI,eNKI,57.0,F,49.7579,sub-A00037386_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00037386/ses-DS2/anat/sub-A00037386_ses-DS2_T1w.nii.gz,224f6e6b90853f6c7721386afbe75afe,5707746\r\neNKI_A00038411,eNKI,eNKI,44.0,F,42.8667,sub-A00038411_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00038411/ses-DS2/anat/sub-A00038411_ses-DS2_T1w.nii.gz,aa31b8aa4ce3c8dc76eda2d4b32e42e1,5891211\r\neNKI_A00038806,eNKI,eNKI,62.0,F,64.6443,sub-A00038806_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00038806/ses-ALGA/ses-ALGA/anat/sub-A00038806_ses-ALGA_T1w.nii.gz,5a1fc855780dbf3c69de88191044ebd8,7388162\r\neNKI_A00038998,eNKI,eNKI,26.0,M,48.5377,sub-A00038998_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00038998/ses-DS2/anat/sub-A00038998_ses-DS2_T1w.nii.gz,403daadc20344fb88f45573bc7b6c1a6,6429508\r\neNKI_A00039655,eNKI,eNKI,69.0,M,67.7532,sub-A00039655_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00039655/ses-DS2/anat/sub-A00039655_ses-DS2_T1w.nii.gz,0018fe999a51a76ca63a5eaaff78ef5a,6338268\r\neNKI_A00039657,eNKI,eNKI,83.0,F,67.5491,sub-A00039657_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00039657/ses-DS2/anat/sub-A00039657_ses-DS2_T1w.nii.gz,f6f7ad8e81efb53245df475894a1d8a2,4643193\r\neNKI_A00040181,eNKI,eNKI,85.0,F,74.5461,sub-A00040181_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00040181/ses-DS2/anat/sub-A00040181_ses-DS2_T1w.nii.gz,69e7e04e07f43157b1ae8f8a6b1328ca,5556196\r\neNKI_A00040324,eNKI,eNKI,70.0,M,68.5493,sub-A00040324_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00040324/ses-DS2/anat/sub-A00040324_ses-DS2_T1w.nii.gz,b3dd8be596e3f0f92aa2266cb681c55b,5525556\r\neNKI_A00040462,eNKI,eNKI,47.0,F,50.6175,sub-A00040462_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00040462/ses-DS2/anat/sub-A00040462_ses-DS2_T1w.nii.gz,a3e5e1acf3aa8f176e40405b446229d2,5669470\r\neNKI_A00040525,eNKI,eNKI,43.0,F,49.2565,sub-A00040525_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00040525/ses-DS2/anat/sub-A00040525_ses-DS2_T1w.nii.gz,c690dae2f44e20c47b973bfdd08b7143,5650295\r\neNKI_A00040567,eNKI,eNKI,65.0,F,51.0094,sub-A00040567_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00040567/ses-DS2/anat/sub-A00040567_ses-DS2_T1w.nii.gz,fb30898ec76a90ce371eba142d1b6749,5305750\r\neNKI_A00043417,eNKI,eNKI,53.0,F,58.8471,sub-A00043417_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00043417/ses-DS2/anat/sub-A00043417_ses-DS2_T1w.nii.gz,9806f3e0032891c32483bc7dc3fb7b22,4962963\r\neNKI_A00043521,eNKI,eNKI,44.0,F,38.686,sub-A00043521_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00043521/ses-DS2/anat/sub-A00043521_ses-DS2_T1w.nii.gz,fe8296d02194b5c092255367e6fe8822,5329740\r\neNKI_A00043635,eNKI,eNKI,50.0,M,55.7051,sub-A00043635_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00043635/ses-DS2/anat/sub-A00043635_ses-DS2_T1w.nii.gz,574914e91c4f194915f303d366b2c9fd,5997354\r\neNKI_A00043739,eNKI,eNKI,49.0,F,46.1893,sub-A00043739_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00043739/ses-DS2/anat/sub-A00043739_ses-DS2_T1w.nii.gz,04348189ffb0572ea0e872f4b1eae61c,5204205\r\neNKI_A00050795,eNKI,eNKI,22.0,M,38.0612,sub-A00050795_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00050795/ses-DS2/anat/sub-A00050795_ses-DS2_T1w.nii.gz,0ff611c6d28bf912ee02b66b81f7e7a0,5510029\r\neNKI_A00051514,eNKI,eNKI,21.0,F,25.982,sub-A00051514_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00051514/ses-DS2/anat/sub-A00051514_ses-DS2_T1w.nii.gz,4abc613d79e7211340e57bc8798de6a2,5679017\r\neNKI_A00051676,eNKI,eNKI,32.0,M,43.0932,sub-A00051676_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00051676/ses-DS2/anat/sub-A00051676_ses-DS2_T1w.nii.gz,975e1d4fbeabac5b78352418efd61abb,6320519\r\neNKI_A00051678,eNKI,eNKI,66.0,F,62.7419,sub-A00051678_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00051678/ses-DS2/anat/sub-A00051678_ses-DS2_T1w.nii.gz,a7bf9f5e6747148e5ee153a85b7634b0,5302774\r\neNKI_A00051679,eNKI,eNKI,27.0,M,20.3922,sub-A00051679_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00051679/ses-DS2/anat/sub-A00051679_ses-DS2_T1w.nii.gz,88df32a86e376963ab7e56e90d3aac79,6272611\r\neNKI_A00052340,eNKI,eNKI,33.0,F,36.5988,sub-A00052340_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00052340/ses-NFB2/anat/sub-A00052340_ses-NFB2_T1w.nii.gz,eda2b25eedae950be1246b9639653dff,5842349\r\neNKI_A00054895,eNKI,eNKI,59.0,F,55.1932,sub-A00054895_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00054895/ses-ALGA/ses-ALGA/anat/sub-A00054895_ses-ALGA_T1w.nii.gz,267a1b00218d3b427129400888819357,6252676\r\neNKI_A00059664,eNKI,eNKI,63.0,F,67.7314,sub-A00059664_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00059664/ses-DS2/anat/sub-A00059664_ses-DS2_T1w.nii.gz,447910176aa13392c30bcbe22800d545,5006962\r\neNKI_A00060925,eNKI,eNKI,24.0,F,31.4202,sub-A00060925_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00060925/ses-DS2/anat/sub-A00060925_ses-DS2_T1w.nii.gz,68447fda9fe445e04226610dcca0f9cc,5708825\r\neNKI_A00061203,eNKI,eNKI,53.0,F,55.0085,sub-A00061203_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00061203/ses-DSA/anat/sub-A00061203_ses-DSA_T1w.nii.gz,d4613f351f8fef319aaf4786e2ec428e,9909902\r\neNKI_A00061485,eNKI,eNKI,64.0,F,62.6683,sub-A00061485_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00061485/ses-DSA/anat/sub-A00061485_ses-DSA_T1w.nii.gz,0509dde40d2549e474067c586baee841,9029209\r\neNKI_A00062285,eNKI,eNKI,51.0,F,49.2669,sub-A00062285_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00062285/ses-ALGA/ses-ALGA/anat/sub-A00062285_ses-ALGA_T1w.nii.gz,36eb8cbaae3dc207c5b8b88732014711,6626352\r\neNKI_A00064328,eNKI,eNKI,69.0,F,67.1149,sub-A00064328_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00064328/ses-ALGAFU1/ses-ALGAFU1/anat/sub-A00064328_ses-ALGAFU1_T1w.nii.gz,fa75387575925afe719c11a7aa5cf4cb,7147334\r\neNKI_A00065995,eNKI,eNKI,45.0,M,52.8531,sub-A00065995_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00065995/ses-ALGAFU1/ses-ALGAFU1/anat/sub-A00065995_ses-ALGAFU1_T1w.nii.gz,1bbcbb7862cd0ab4702d560a8adf5335,7444300\r\neNKI_A00066827,eNKI,eNKI,26.0,F,28.4708,sub-A00066827_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00066827/ses-NFB3/anat/sub-A00066827_ses-NFB3_T1w.nii.gz,677e94a7d82a815053423eba1f9c6ff6,11579442\r\neNKI_A00066865,eNKI,eNKI,36.0,F,36.1428,sub-A00066865_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00066865/ses-NFBA/anat/sub-A00066865_ses-NFBA_T1w.nii.gz,4821811c26b4a5df8e30d8e65603ffa6,8733361\r\neNKI_A00073330,eNKI,eNKI,35.0,F,39.845,sub-A00073330_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00073330/ses-NFBA/anat/sub-A00073330_ses-NFBA_T1w.nii.gz,d1193415a888892c92b86a6b3c6a7f52,8952652\r\neNKI_A00087985,eNKI,eNKI,59.0,F,47.7338,sub-A00087985_T1w.nii.gz,https://fcp-indi.s3.amazonaws.com/data/Projects/RocklandSample/RawDataBIDS/sub-A00087985/ses-BAS1/anat/sub-A00087985_ses-BAS1_T1w.nii.gz,1b0c0a8ef3d67b1859600f72cf8a4282,7336895\r\n"
REF_JSON = "{\"antsxnet\":{\"brainAgeDeepBrainNet.h5\":\"50d76851567d502e38755892746473af\",\"brainExtractionRobustT1.h5\":\"515a171b701df6ab512735f02ea26b14\",\"croppedMni152.nii.gz\":\"2170efb430dcc82dc97860985c8f18df\",\"croppedMni152Priors.nii.gz\":\"5e8e851dc2f99cc9b4aac634b4590be0\",\"sixTissueOctantBrainSegmentationWithPriors1.h5\":\"61d5c2574913f3f2e4716fd0eb04a43a\",\"S_template3.nii.gz\":\"ba473fc2a89895be028ec798842d92b7\"},\"bang_weights\":{\"BrainAge_1.pth\":\"b19d69ebbf3dc46e27bd4215b890e03f\",\"BrainAge_2.pth\":\"8ed0e327a7b4d06c92c0884f08edfd0a\",\"BrainAge_3.pth\":\"9b4f95fdd0fdda5fecfc72fa8cdbeb38\",\"BrainAge_4.pth\":\"1abf4a1019d22b9912511a494c794336\",\"BrainAge_5.pth\":\"b0eee326c4494a3294809ef5b2749626\"},\"bang_code_md5\":\"fed48108580f032f95f8d1ed1505a461\",\"dbn_ref\":{\"IXI002\":{\"pred\":36.2744,\"gap_mean\":[0.00034,0.0,0.00328,0.00124,0.00589,0.0,0.0,0.0,0.0,0.00033,0.0,0.00083,0.0001,0.0,0.0046,0.00175,0.0,1e-05,0.07146,3e-05,0.00047,0.0,0.00982,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00158,0.0,0.0,0.00117,2e-05,0.02221,0.00072,0.00724,0.0,0.0,0.00112,0.00099,0.21199,0.0,0.0,0.0,0.0,1e-05,0.01098,0.0,0.0,0.00921,0.002,0.0,0.00029,0.10151,2e-05,0.0,0.0,0.0,0.0,0.0,0.00413,0.0,0.0,0.0,0.0,0.00282,0.0,0.00016,0.0,0.0,0.00078,0.0,0.00952,0.0027,0.12466,0.0,1e-05,0.0,0.0,0.0,0.0,0.00015,0.0,0.04128,0.00529,0.0,0.0,0.03921,7e-05,0.0,0.00811,4e-05,0.0,0.0,0.00022,1e-05,0.0,0.0,0.0,0.23805,5e-05,0.0,6e-05,0.0,1e-05,0.0,0.00352,0.0,0.0,0.0,0.0,0.92945,0.0,0.0,0.01167,6e-05,0.00012,0.07527,0.01048,0.0,0.0,0.11992,0.00013,0.0,9e-05,0.0,0.00039,0.0,0.0,0.00024,0.02465,0.00037,0.0,0.00231,8e-05,0.0,0.00021,0.0,9e-05,0.00011,0.0,0.00017,0.00183,0.0,0.00632,0.00026,0.0,0.0,0.00643,0.0,0.00099,0.0001,0.0,5.6917,0.001,0.01247,0.0,0.0,0.0,0.00026,0.0,0.2672,0.0011,0.00027,0.01026,0.0,0.0,0.0,0.0,0.00014,0.0,0.0,0.0,7e-05,0.0,0.00043,0.0,0.0021,0.0,0.00231,0.00022,0.0,0.0,7e-05,0.00413,0.0,0.0,0.00832,0.00476,0.0,0.0,0.0,0.00077,0.0,0.0,0.0,0.0029,0.00029,0.01039,0.0,0.00237,0.00175,0.00014,1e-05,0.0,0.0,0.00854,0.00013,0.00054,0.0,0.03746,0.00031,0.0,0.0,0.00098,0.0,0.00021,0.0,0.0,0.0,0.0,0.03152,0.00047,0.02171,0.00097,0.0,0.02747,0.00015,0.00158,0.01113,0.0001,2e-05,0.0,0.001,0.00406,0.00097,0.0,0.0,0.0,0.00066,0.0,0.0,0.0,0.0,0.0,0.00461,0.0001,0.0,0.0,0.00152,0.0,0.0158,0.04758,0.0,0.0,0.0,0.0,0.01699,0.00041,0.0,0.0,0.00301,0.00013,0.0,0.0,0.0,4e-05,0.10139,0.0,0.00119,0.00012,0.0,0.0,0.0023,0.16661,0.01059,0.00503,0.0,0.0,0.00314,5e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00611,0.00028,0.0,0.0,0.0,0.00023,0.0,0.00056,0.0,0.01305,0.00354,0.0,0.0,0.00131,0.00024,0.0,0.0,0.00066,0.0005,0.0,0.0,0.0,0.85363,0.0,0.0,2e-05,0.0,0.0,0.00058,0.00071,0.0,0.00822,0.0034,0.0,0.0,0.08796,0.00085,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00055,0.0,3e-05,0.0,0.0,0.01281,1e-05,0.00019,0.0,0.00206,0.0,0.0,0.00119,0.00058,0.0,0.24087,0.0,0.00058,3e-05,0.0,0.00164,0.0,0.00017,0.00204,0.0,0.0,0.0,0.0,0.0,0.00022,0.0,0.0,0.00071,4e-05,0.04649,0.00069,0.0,0.0,0.0,0.0,0.0,0.0,0.0043,0.06091,0.0,0.06174,0.0,0.00042,0.0,0.00081,0.00031,0.00039,0.0,0.14739,0.00011,0.00855,0.0,0.0003,0.0,0.0,0.00046,0.0,0.00033,0.09123,0.0,0.0,0.00089,0.0,0.00096,0.0,0.0,2e-05,0.00073,0.0,0.02193,0.0,0.0,6e-05,0.0,0.0008,0.02242,0.0,0.00834,0.00015,3e-05,0.00021,0.01311,0.00019,0.0,0.0,0.0,0.00013,7e-05,0.0,0.0,0.00103,0.0,0.00142,0.0,0.00396,0.00668,0.35809,0.00142,0.0,1.59413,0.00512,0.0,0.00384,0.0001,0.0,0.00063,0.0,0.00728,0.00041,0.0,0.00066,0.0002,0.00021,0.0,0.0012,0.00135,2e-05,0.29486,0.0008,0.0,1e-05,0.01375,0.0,0.13199,0.00261,3e-05,0.0,0.0001,0.00643,0.0,0.0,0.05177,0.00352,0.0,0.0,0.00067,0.0,0.0,0.00108,0.0,0.0,0.0,0.00017,6e-05,0.00576,0.0,0.0,0.0026,0.0,0.0,0.0,0.00023,0.0,0.00055,0.0,0.0,0.00029,0.0,0.00578,0.0,0.13897,0.0,0.0,0.0,1e-05,0.0,0.0,0.0,0.11,0.00032]},\"IXI012\":{\"pred\":46.2345,\"gap_mean\":[0.00134,0.0,0.0211,0.00474,0.0397,0.0,0.0,0.0,2e-05,0.00337,0.0,0.01028,3e-05,0.0,0.00339,0.01158,1e-05,2e-05,0.04827,0.00026,0.00392,0.0,0.00045,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.017,0.0,0.0,0.00013,0.0,0.00586,0.00042,0.00651,0.0,0.0,0.00085,0.00075,0.28359,0.0,0.0,0.0011,0.0,0.0,0.02952,0.00032,0.0,0.04437,0.0004,0.0,0.00025,0.15952,0.00036,0.0,0.0,0.0,0.00011,0.0,0.0103,0.0,0.0,0.0,0.0,0.0218,0.0,0.00045,0.00018,0.00064,3e-05,0.0,0.00132,0.00181,0.18508,0.0,2e-05,0.0,0.0,2e-05,0.0,2e-05,0.0,0.02563,0.03697,0.0,0.0,0.06122,0.00043,0.0,0.02859,2e-05,0.00018,0.0,0.0011,0.0,0.0,0.00044,0.0,0.32811,0.00093,0.0,0.0,0.00267,0.0,7e-05,0.00196,0.0,0.0,6e-05,0.00024,1.16034,0.0,0.0,0.00735,3e-05,0.00031,0.16281,0.0062,0.0,0.00057,0.17241,0.00786,1e-05,0.00038,3e-05,0.00045,0.0,0.0,0.00091,0.01183,0.00328,0.0,0.0006,0.0,0.00042,0.0018,0.0,0.0,0.0,0.0,0.00058,0.00042,0.0,0.00669,0.00029,0.0,0.0,0.00144,0.00022,0.00643,4e-05,0.00089,6.96401,0.00387,0.01092,0.0,0.0,0.0,0.00102,0.0,0.36523,0.0,0.0,0.03296,0.00014,0.0,0.0,0.0,0.00262,0.00076,0.0,0.0,0.0001,2e-05,0.00036,0.00089,0.00573,0.0,0.00677,0.00233,0.0,0.0001,0.00251,0.00367,0.0,0.0,0.00806,0.00302,0.0,0.0,0.0,0.00017,0.00023,0.0,0.0,0.00143,0.00088,0.01536,0.0,0.00198,0.00528,0.00012,0.0,0.00048,6e-05,0.01293,0.0,0.00102,0.0001,0.03182,9e-05,0.0,0.0,0.01259,0.0,0.00333,4e-05,0.00352,0.0,0.0,0.02074,0.00068,0.0727,0.00905,0.00043,0.02588,0.00091,0.00151,0.00924,4e-05,0.0,0.0,0.00513,0.01125,0.00115,0.00182,0.0,0.0,0.00026,6e-05,0.0,0.0,0.0,0.00048,0.00285,0.0,1e-05,0.0,0.00393,0.0,0.02033,0.10065,0.0,0.0,0.0,0.0,0.03384,0.00093,0.0,0.00018,0.01276,0.00252,0.0,1e-05,0.0,8e-05,0.17421,0.0,0.00243,0.00088,9e-05,0.0,0.00143,0.21939,0.02896,0.00709,0.00017,0.00032,0.00503,0.0023,0.0,0.0,0.00032,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00832,0.0,0.00021,0.0,0.0,0.00468,0.0,0.00087,0.00231,0.01039,0.02222,0.0,0.00514,0.00046,0.00017,0.0,0.00043,0.00303,0.00564,0.0,0.0,0.0,1.07336,0.0,0.0,7e-05,7e-05,0.0,0.00125,0.00017,0.0,0.00346,0.00361,0.0,0.0,0.1308,0.00047,0.00397,0.00011,0.0001,0.0,0.0,0.0,0.0,0.00249,0.0,0.0,0.0,0.00012,0.04509,8e-05,0.01043,0.0,0.00342,0.0,0.0,0.00374,0.0054,0.0,0.34044,0.0,0.00038,0.00182,0.00021,0.00189,0.0,0.00127,0.00623,0.0,0.0,0.0,0.00011,3e-05,0.00085,0.0,0.0,0.01134,0.00016,0.09269,0.00249,0.00528,0.0,0.00026,0.0,0.00023,0.0,0.00256,0.06498,0.00012,0.08693,0.0,0.00125,0.0,0.00696,0.00195,0.00122,0.0,0.24081,0.00428,0.00326,1e-05,0.00184,0.00122,0.0,0.0,5e-05,0.0016,0.07348,0.0,0.0,0.00018,0.0,0.00067,0.00012,0.0,0.0045,0.00789,0.0,0.02908,0.0,0.0,1e-05,0.0,0.00155,0.0241,0.0,0.04368,0.0,0.00169,0.00157,0.01026,0.00041,0.0,0.0,0.0,0.00167,0.00028,2e-05,0.0,0.00493,0.0,0.0053,0.0,0.00972,0.00953,0.39284,0.00751,0.0,1.98309,0.00956,0.00265,0.01491,0.0,1e-05,5e-05,0.0,0.0081,0.00141,0.0,0.00213,0.0,0.0004,0.0,0.00124,0.0001,0.0,0.169,0.00311,0.0,0.0,0.03671,0.0,0.13147,0.01714,1e-05,0.0,0.0,0.00667,0.0,0.0,0.0412,0.00943,0.0,0.0,0.0059,0.0,0.0,0.00041,0.0,0.0,6e-05,0.00721,0.00056,0.02652,0.0,0.0,0.01757,1e-05,0.0,0.0,0.00187,0.0,0.00122,0.0,0.0,0.00141,0.0,0.0156,0.0,0.20339,0.0,0.0,0.0,0.00178,0.0,2e-05,7e-05,0.15742,0.00968]},\"IXI013\":{\"pred\":49.2911,\"gap_mean\":[0.00287,0.0,0.03498,0.00962,0.02891,7e-05,0.00011,0.0,5e-05,8e-05,0.0,0.00223,0.00025,0.0,0.00824,0.01837,0.0,0.0,0.11683,0.00019,0.00342,8e-05,0.00155,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.01948,0.0,0.0,1e-05,0.0,0.03958,0.00024,0.0129,0.0,0.0,0.00603,0.00036,0.32952,0.00019,0.0,0.00157,0.0,0.0,0.03886,0.00012,0.0,0.0808,0.00149,0.0,0.00095,0.18738,0.00065,7e-05,0.0,0.0,0.0,0.0,0.01147,0.0,0.0,0.0,0.0,0.01888,0.0,0.00012,0.0,1e-05,0.00057,0.0,0.00154,0.00329,0.22475,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8e-05,0.01612,0.064,2e-05,0.0,0.07117,0.0,0.0,0.01968,0.00077,0.00039,0.0,0.00187,0.00011,0.0,0.00011,0.0,0.37583,0.0033,0.0,0.0,0.00331,1e-05,0.0,0.00098,0.0,0.0,9e-05,0.0,1.26055,1e-05,0.00037,0.00898,0.0,0.00349,0.20993,0.0168,0.0,0.00059,0.21318,0.0262,0.00013,0.00017,0.00075,0.00097,0.0,0.0,0.00206,0.00526,0.00526,0.0,0.00168,0.0,0.00033,0.00593,0.0,0.001,0.0,0.0,0.00394,0.00488,0.00022,0.00794,0.00374,0.0,0.0,0.00013,0.002,0.00516,0.0,0.00108,7.40337,0.00398,0.02736,0.0,0.0,0.0,0.00055,0.0,0.42409,0.00067,5e-05,0.04935,0.00026,0.0,0.0,0.0,0.014,0.0,0.00018,0.0001,0.0,0.0,0.00213,0.0,0.01618,0.0,0.00301,0.00731,0.0,0.00662,0.004,0.00358,0.0,0.0,0.03083,0.00446,0.0,0.00023,0.0,0.00118,0.0,0.0,0.0,0.00345,0.00092,0.01993,0.0,0.00565,0.00452,0.00014,0.0,0.0,0.00096,0.00732,0.0,0.00687,0.00017,0.06487,2e-05,5e-05,0.0,0.03994,0.0,0.00171,0.00018,0.00071,0.00025,1e-05,0.0678,0.00526,0.11318,0.02075,0.0,0.03452,0.00147,0.00427,0.03073,0.0,0.00011,0.0,0.01812,0.02701,0.00247,0.0,8e-05,0.0,0.00149,0.0,0.0,0.0,0.0,0.0,0.00095,0.00011,0.0,0.0011,0.00699,0.00052,0.02895,0.13276,0.0,0.0,0.0,0.0,0.05054,0.00357,0.00062,5e-05,0.01043,0.00235,0.0,0.0,0.0,0.0,0.22727,8e-05,0.00388,5e-05,0.0,0.0,0.00136,0.25071,0.02467,0.01733,0.00109,6e-05,0.02745,0.00414,0.0,0.0,7e-05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0215,1e-05,0.0,0.0,0.00026,0.00942,0.0,0.00558,0.00108,0.01132,0.04564,0.0,0.00037,0.00041,3e-05,0.0,0.00268,0.01435,0.02716,0.0,0.0,0.0,1.15905,0.00017,0.0,0.0059,0.00097,0.0,0.00753,0.00023,0.0,0.00878,0.00214,1e-05,0.0001,0.15973,0.00026,0.03455,0.0,0.0,0.0003,0.0,0.0,0.0,0.00239,0.0,0.0,7e-05,0.00111,0.06508,0.00023,0.02817,2e-05,0.00689,0.00013,0.0,0.00688,0.00176,0.0,0.39411,0.0,0.00325,0.00135,0.0,0.00358,0.0,0.00203,0.00563,0.0,0.0,0.0,0.00013,0.0,0.00422,0.0,0.00012,0.01839,0.00011,0.1174,0.00736,0.00258,0.0,0.0,0.0,0.00035,0.0,0.00011,0.07184,0.0,0.10388,0.0,0.00139,0.0,0.01213,0.00281,0.00182,0.00017,0.28447,0.00323,0.00332,0.00029,0.00015,2e-05,7e-05,0.0,0.0,0.00331,0.09681,2e-05,0.0,0.00047,3e-05,0.00044,0.00091,0.00564,0.00199,0.0005,0.0,0.03908,0.0,0.0,0.0,0.0,0.00109,0.0295,0.0,0.04376,0.0,0.03618,0.00493,0.02236,0.00342,0.0,0.0,0.0,0.00417,5e-05,2e-05,0.0,0.00307,0.0,0.00471,0.0,0.00629,0.01761,0.47712,0.01,0.0,2.13708,0.00881,0.00371,0.02376,0.0,6e-05,0.00028,0.00168,0.01095,0.00917,0.0,0.0021,0.01454,0.00125,0.0,0.00192,0.00487,0.0,0.16982,0.01108,0.00301,0.0,0.03184,0.0,0.13598,0.02462,0.00018,0.0,0.00038,0.03176,0.0,0.0,0.10259,0.02037,0.0,0.0,0.00491,0.0,0.0,0.0067,0.0,0.0,0.0,0.0105,0.00132,0.03898,0.0,0.0,0.02242,8e-05,0.0,0.0,0.0026,0.0,0.00161,0.0,0.00104,0.00152,0.0,0.02874,0.0,0.23809,0.0,0.0,0.0,0.0023,9e-05,0.0,0.00016,0.18081,0.03084]}},\"bang_ref\":{\"IXI002\":{\"as_is\":37.2,\"flip_LR\":35.5,\"flip_AP\":40.1,\"flip_SI\":42.7}}}"

In [ ]:
# ===== 3. Drive, logging, safety helpers =====
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import io, json, time, shutil, hashlib, pathlib, datetime
import numpy as np, pandas as pd, requests
from google.colab import drive, runtime
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive') / DRIVE_FOLDER
OUT = DRIVE / 'outputs'
DBN_OUT, BANG_OUT, CACHE = OUT / 'dbn_embed', OUT / 'bang_embed', DRIVE / 'ANTsXNet'
WORK = pathlib.Path('/content/work')
for d in (DBN_OUT, BANG_OUT, CACHE, WORK / 't1', WORK / 'bang'):
    d.mkdir(parents=True, exist_ok=True)
LOG = OUT / 'run_log.txt'

manifest = pd.read_csv(io.StringIO(MANIFEST_CSV))
REF = json.loads(REF_JSON)
ROW = {r.scan_id: r for r in manifest.itertuples()}

def log(msg):
    line = f'{datetime.datetime.now():%m-%d %H:%M:%S}  {msg}'
    print(line, flush=True)
    with open(LOG, 'a') as f:
        f.write(line + '\n')

def release():
    if AUTO_RELEASE_RUNTIME:
        log('releasing the Colab runtime (no more compute is used)')
        try:
            drive.flush_and_unmount()
        finally:
            runtime.unassign()

def stop(reason, ok=False):
    log(('PAUSED: ' if ok else 'STOPPED: ') + reason)
    if not ok:
        (OUT / 'STOPPED.txt').write_text(f'{datetime.datetime.now()}\n{reason}\n')
    release()
    raise Stop(reason)

def md5sum(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

def fetch(url, dest, md5=None, tries=5):
    # download to .part, verify MD5, then rename; retries with backoff
    dest = pathlib.Path(dest)
    if dest.exists() and (md5 is None or md5sum(dest) == md5):
        return dest
    last = None
    for i in range(tries):
        try:
            tmp = dest.with_name(dest.name + '.part')
            with requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for b in r.iter_content(1 << 20):
                        f.write(b)
            if md5 and md5sum(tmp) != md5:
                raise IOError(f'MD5 mismatch for {dest.name} (server file differs from the laptop copy)')
            os.replace(tmp, dest)
            return dest
        except Exception as e:
            last = e
            time.sleep(10 * (i + 1))
    raise RuntimeError(f'download failed after {tries} tries: {url}  ({last!r})')

def put(src, dst):
    # copy to Drive under a temp name, then rename: a disconnect never leaves a half-written file
    tmp = dst.with_name(dst.name + '.part')
    shutil.copyfile(src, tmp)
    os.replace(tmp, dst)

def dbn_ok(p):
    try:
        d = np.load(p)
        return (d['gap'].shape == (80, 512) and d['dense'].shape == (80, 1024)
                and np.isfinite(d['gap']).all() and np.isfinite(d['dense']).all()
                and 0 < float(d['predicted_age']) < 120)
    except Exception:
        return False

def bang_ok(p):
    try:
        d = np.load(p)
        return d['feat'].shape == (5, 512) and np.isfinite(d['feat']).all() and np.isfinite(d['preds']).all()
    except Exception:
        return False

import torch
GPU = torch.cuda.is_available()
if (OUT / 'STOPPED.txt').exists():
    log('note: the previous session stopped with: ' + (OUT / 'STOPPED.txt').read_text().strip().replace('\n', ' | '))
    (OUT / 'STOPPED.txt').unlink()
free_gb = shutil.disk_usage('/content/drive/MyDrive').free / 1e9
log(f'session start  runtime={"T4 GPU" if GPU else "CPU"}  drive_free={free_gb:.1f} GB  scans={len(manifest)}')
if free_gb < 5:
    stop(f'Google Drive has only {free_gb:.1f} GB free; about 4 GB is needed for outputs.')

In [ ]:
# ===== 4. Load models + pre-flight reproduction checks =====
from concurrent.futures import ThreadPoolExecutor
N = len(manifest)
D1 = {s for s in manifest.scan_id if (DBN_OUT / f'{s}.npz').exists()}    # DeepBrainNet done
D2 = {s for s in manifest.scan_id if (BANG_OUT / f'{s}.npz').exists()}   # BrainAgeNeXt done
log(f'DeepBrainNet {len(D1)}/{N} done   BrainAgeNeXt {len(D2)}/{N} done')
NEED1, NEED2 = len(D1) < N, len(D2) < N
BANG_READY = GPU and NEED2

def t1_of(r):
    return fetch(r.url, WORK / 't1' / r.fname, r.md5)

if NEED1:
    import tensorflow as tf
    for g in tf.config.list_physical_devices('GPU'):
        tf.config.experimental.set_memory_growth(g, True)    # leave GPU memory for BrainAgeNeXt
    import ants, tf_keras as keras
    from antspynet.utilities import preprocess_brain_image, set_antsxnet_cache_directory, get_pretrained_network
    set_antsxnet_cache_directory(str(CACHE))
    h5 = get_pretrained_network('brainAgeDeepBrainNet')
    if md5sum(h5) != REF['antsxnet']['brainAgeDeepBrainNet.h5']:
        stop('DeepBrainNet weights downloaded by antspynet differ from the laptop copy.')
    model = keras.models.load_model(h5)
    feat_model = keras.Model(model.input, [model.get_layer('global_average_pooling2d_2').output,
                                           model.get_layer('dense_3').output, model.output])
    SLICES = list(range(45, 125))

    def dbn_scan(r, t1_path, save=True):
        prep = preprocess_brain_image(ants.image_read(str(t1_path)), truncate_intensity=(0.01, 0.99),
                                      brain_extraction_modality='t1', template='croppedMni152',
                                      template_transform_type='antsRegistrationSyNQuickRepro[a]',
                                      do_bias_correction=True, do_denoising=True, verbose=False)
        img = prep['preprocessed_image'] * prep['brain_mask']
        img = (img - img.min()) / (img.max() - img.min())
        batch = np.zeros((len(SLICES), *img.shape[0:2], 3), dtype=np.float32)
        for j, idx in enumerate(SLICES):
            batch[j] = ants.slice_image(img, axis=2, idx=idx).numpy()[..., None]
        gap, dense, sp = feat_model.predict(batch, verbose=0)
        pred = float(np.median(sp[:, 0]))
        if save:
            lp, ln = WORK / f'{r.scan_id}_prep.nii.gz', WORK / f'{r.scan_id}.npz'
            ants.image_write(img, str(lp))
            np.savez_compressed(ln, gap=gap.astype(np.float16), dense=dense.astype(np.float16),
                                slice_pred=sp[:, 0].astype(np.float32), predicted_age=np.float32(pred))
            if not dbn_ok(ln):
                raise ValueError('saved features failed the shape/NaN check')
            put(lp, DBN_OUT / lp.name)
            put(ln, DBN_OUT / ln.name)          # npz last: it is what marks the scan as done
            lp.unlink(); ln.unlink()
            D1.add(r.scan_id)
        return pred, gap.astype(np.float32).mean(0)

    def check_ref(sid, pred, gmean):
        ref = REF['dbn_ref'][sid]
        dp = abs(pred - ref['pred'])
        c = float(np.corrcoef(gmean, np.array(ref['gap_mean']))[0, 1])
        log(f'  reproduce {sid}: colab={pred:.2f} laptop={ref["pred"]:.2f} (diff {dp:.2f} y)  feature r={c:.4f}')
        if dp > 1.0 or c < 0.98:
            stop(f'pre-flight failed on {sid}: Colab does not reproduce the laptop result '
                 f'(diff {dp:.2f} y, feature r={c:.3f}). Nothing else was run.')

    # re-run IXI002 fresh every session (Colab's preinstalled packages can change between sessions)
    log('pre-flight: re-running IXI002 through DeepBrainNet and comparing with the laptop ...')
    r0 = ROW['IXI002']
    pred0, g0 = dbn_scan(r0, t1_of(r0), save='IXI002' not in D1)
    check_ref('IXI002', pred0, g0)
    for f in CACHE.iterdir():
        if f.name in REF['antsxnet'] and md5sum(f) != REF['antsxnet'][f.name]:
            stop(f'antspynet file {f.name} differs from the laptop copy.')
    log('DeepBrainNet pre-flight passed')

VARIANTS = {'as_is': None, 'flip_LR': 2, 'flip_AP': 3, 'flip_SI': 4}   # tensor is (1, 1, X, Y, Z)
LOCK = OUT / 'bang_orientation.json'
ORIENT = json.loads(LOCK.read_text())['orientation'] if LOCK.exists() else None

if BANG_READY:
    B = WORK / 'bang'
    fetch('https://raw.githubusercontent.com/FrancescoLR/BrainAgeNeXt/main/BrainAge_estimation.py',
          B / 'BrainAge_estimation.py', REF['bang_code_md5'])
    for name, h in REF['bang_weights'].items():
        fetch(f'https://huggingface.co/FrancescoLR/BrainAgeNeXt/resolve/main/{name}', B / name, h)
    sys.path.insert(0, str(B))
    from BrainAge_estimation import MedNeXtEncReg, prepare_transforms   # official file, MD5-verified
    bang = []
    for i in range(1, 6):
        m = MedNeXtEncReg().cuda()
        m.load_state_dict(torch.load(B / f'BrainAge_{i}.pth', map_location='cuda'))
        bang.append(m.eval())
    tfm = prepare_transforms()

    def bang_run(prep_path, flip_dim):
        x = torch.as_tensor(np.asarray(tfm({'image': str(prep_path)})['image']))[None].cuda()
        if flip_dim is not None:
            x = torch.flip(x, dims=[flip_dim])
        feats, preds = [], []
        with torch.no_grad():
            for m in bang:
                f = torch.flatten(m.global_avg_pool(m.mednextv1(x)), 1)   # same path as forward()
                feats.append(f[0].float().cpu().numpy())
                preds.append(float(m.regression_fc(f).squeeze()))
        return np.stack(feats), np.array(preds, dtype=np.float32)

    log('pre-flight: BrainAgeNeXt on IXI002, all 4 orientations, vs the laptop ...')
    for k, ref in REF['bang_ref']['IXI002'].items():
        got = float(np.median(bang_run(DBN_OUT / 'IXI002_prep.nii.gz', VARIANTS[k])[1]))
        log(f'  reproduce BrainAgeNeXt IXI002 {k}: colab={got:.1f} laptop={ref:.1f}')
        if abs(got - ref) > 1.5:
            stop(f'BrainAgeNeXt on Colab does not reproduce the laptop ({k}: {got:.1f} vs {ref:.1f}).')
    log('BrainAgeNeXt pre-flight passed' + (f'; orientation locked earlier: {ORIENT}' if ORIENT else ''))

In [ ]:
# ===== 5. Main loop: each scan -> DeepBrainNet -> BrainAgeNeXt =====
class Guard:
    def __init__(self, name):
        self.name, self.fail, self.row = name, 0, 0
    def ok(self):
        self.row = 0
    def failed(self, sid, e):
        self.fail += 1
        self.row += 1
        log(f'  {sid} {self.name} FAILED: {e!r}')
        if self.row >= 3:
            stop(f'3 scans in a row failed in {self.name} (last: {sid}: {e!r}).')
        if self.fail > max(5, 0.05 * N):
            stop(f'{self.fail} scans failed in {self.name} (>5%).')

g1, g2, diffs, errs = Guard('DeepBrainNet'), Guard('BrainAgeNeXt'), [], []
GH = [r.scan_id for r in manifest.itertuples() if r.site in ('Guys', 'HH')]

def do_dbn(r, t1):
    pred, g = dbn_scan(r, t1)
    if r.scan_id in REF['dbn_ref']:
        check_ref(r.scan_id, pred, g)
    d = abs(pred - r.original_pred)
    diffs.append(d)
    if d > 8:
        stop(f'{r.scan_id}: DeepBrainNet {pred:.1f} vs laptop original {r.original_pred:.1f} '
             f'({d:.1f} y apart) - preprocessing has drifted.')
    if len(diffs) >= 10 and np.median(diffs[-10:]) > 2.0:
        stop(f'median gap to the laptop predictions over the last 10 scans is '
             f'{np.median(diffs[-10:]):.2f} y (>2) - preprocessing has drifted.')
    return pred

def choose_orientation():
    # chosen once on 20 Guys/HH scans, then locked in a file so every session uses the same one
    global ORIENT
    sample = [ROW[s] for s in GH if s in D1][:20]
    err = {k: [] for k in VARIANTS}
    for r in sample:
        for k, d in VARIANTS.items():
            err[k].append(abs(float(np.median(bang_run(DBN_OUT / f'{r.scan_id}_prep.nii.gz', d)[1]))
                              - r.chronological_age))
    maes = {k: float(np.mean(v)) for k, v in err.items()}
    log('orientation check (MAE, 20 Guys/HH scans): ' + '  '.join(f'{k}={v:.2f}' for k, v in maes.items()))
    best = min(maes, key=maes.get)
    if maes[best] > 10:
        stop(f'best BrainAgeNeXt orientation still has MAE {maes[best]:.1f} y - input space is wrong, '
             'not worth running on every scan.')
    LOCK.write_text(json.dumps({'orientation': best, 'mae': maes}))
    ORIENT = best
    log(f'orientation locked: {ORIENT}')

def do_bang(r):
    feats, preds = bang_run(DBN_OUT / f'{r.scan_id}_prep.nii.gz', VARIANTS[ORIENT])
    ln = WORK / f'{r.scan_id}_bang.npz'
    np.savez_compressed(ln, feat=feats.astype(np.float16), preds=preds,
                        predicted_age=np.float32(np.median(preds)), orientation=ORIENT)
    if not bang_ok(ln):
        raise ValueError('saved features failed the shape/NaN check')
    put(ln, BANG_OUT / f'{r.scan_id}.npz')
    ln.unlink()
    D2.add(r.scan_id)
    errs.append(abs(float(np.median(preds)) - r.chronological_age))
    if len(errs) >= 20 and np.mean(errs[-20:]) > 12:
        stop(f'BrainAgeNeXt MAE over the last 20 scans is {np.mean(errs[-20:]):.1f} y (>12) - stopping.')
    return float(np.median(preds))

def try_bang(r):
    if not BANG_READY or r.scan_id not in D1 or r.scan_id in D2:
        return None
    if ORIENT is None:
        if sum(s in D1 for s in GH) < 20:
            return None                       # not enough scans yet to pick the orientation
        choose_orientation()
    try:
        p = do_bang(r)
        g2.ok()
        return p
    except Stop:
        raise
    except Exception as e:
        g2.failed(r.scan_id, e)

todo = [r for r in manifest.itertuples() if r.scan_id not in D1 or (BANG_READY and r.scan_id not in D2)]
log(f'{len(todo)} scans to process this session')
pool, futs, t0 = ThreadPoolExecutor(max_workers=3), {}, time.time()
for i, r in enumerate(todo):
    for nxt in todo[i:i + 4]:                 # download the next scans while this one runs
        if nxt.scan_id not in D1 and nxt.scan_id not in futs:
            futs[nxt.scan_id] = pool.submit(t1_of, nxt)
    p1 = p2 = None
    if r.scan_id not in D1:
        try:
            t1 = futs.pop(r.scan_id).result()
            p1 = do_dbn(r, t1)
            t1.unlink(missing_ok=True)
            g1.ok()
        except Stop:
            raise
        except Exception as e:
            g1.failed(r.scan_id, e)
            continue
    p2 = try_bang(r)
    if (i + 1) % 5 == 0:
        rate = (time.time() - t0) / (i + 1)
        log(f'working: DeepBrainNet {len(D1)}/{N}  BrainAgeNeXt {len(D2)}/{N}  '
            f'(last {r.scan_id}: age {r.chronological_age:.1f}'
            + (f', DBN {p1:.1f}' if p1 is not None else '') + (f', BANG {p2:.1f}' if p2 is not None else '')
            + f')  {rate:.0f} s/scan, ~{(len(todo) - i - 1) * rate / 3600:.1f} h left')
pool.shutdown(wait=False)

if BANG_READY:                                # scans whose DeepBrainNet finished before the orientation lock
    backlog = [ROW[s] for s in manifest.scan_id if s in D1 and s not in D2]
    if backlog:
        log(f'BrainAgeNeXt backlog: {len(backlog)} scans')
    for k, r in enumerate(backlog, 1):
        try_bang(r)
        if k % 5 == 0:
            log(f'working: BrainAgeNeXt {len(D2)}/{N}')

log(f'finished this pass: DeepBrainNet {len(D1)}/{N}  BrainAgeNeXt {len(D2)}/{N}')
if N - len(D1) > 0.05 * N:
    stop(f'{N - len(D1)} scans still missing DeepBrainNet features; Run all again to retry them.')
if not GPU:
    stop('DeepBrainNet is complete. Switch to Runtime > Change runtime type > T4 GPU, then Run all.', ok=True)
if N - len(D2) > 0.05 * N:
    stop(f'{N - len(D2)} scans still missing BrainAgeNeXt features; Run all again to retry them.')

## Stage 3 - transfer-learning heads
Ridge heads are trained only on IXI Guys+HH (5-fold out-of-fold MAE reported); eNKI is never used to fit them.

**Caveat - pretraining overlap.** BrainAgeNeXt's own training set includes openBHB (which contains IXI)
and NKI-RS (the same Rockland sample as eNKI). Its sub-4 y errors on these cohorts are therefore not a
held-out result. A clean test needs a cohort BrainAgeNeXt only used for testing (e.g. SALD, DLBS).

In [ ]:
# ===== 6. Stage 3 - transfer-learning heads (train on IXI Guys+HH only) =====
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_val_predict

recs = []
for r in manifest.itertuples():
    a, b = DBN_OUT / f'{r.scan_id}.npz', BANG_OUT / f'{r.scan_id}.npz'
    if not (a.exists() and b.exists() and dbn_ok(a) and bang_ok(b)):
        continue
    a, b = np.load(a), np.load(b)
    recs.append(dict(scan_id=r.scan_id, site=r.site, age=r.chronological_age,
                     dbn_raw=float(a['predicted_age']), bang_raw=float(b['predicted_age']),
                     gap=a['gap'].astype(np.float32).mean(0), dense=a['dense'].astype(np.float32).mean(0),
                     bangf=b['feat'].astype(np.float32).mean(0)))
df = pd.DataFrame(recs)
log(f'stage 3: {len(df)} scans with both feature sets  ' + str(df.groupby('site').size().to_dict()))

tr = df['site'].isin(['Guys', 'HH']).values
mae = lambda p, y: float(np.mean(np.abs(np.asarray(p) - np.asarray(y))))
X = lambda cols, m: np.hstack([np.vstack(df.loc[m, c].to_numpy()) for c in cols])
table = []
for name in ('dbn_raw', 'bang_raw'):
    row = {'model': f'{name} (no head)', 'Guys+HH': mae(df.loc[tr, name], df.loc[tr, 'age'])}
    for s in ('eNKI',):
        k = (df.site == s).values
        row[s] = mae(df.loc[k, name], df.loc[k, 'age'])
    table.append(row)
for name, cols in {'DBN gap512': ['gap'], 'DBN dense1024': ['dense'], 'BANG 512': ['bangf'],
                   'DBN gap + BANG': ['gap', 'bangf']}.items():
    head = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 4, 20)))
    oof = cross_val_predict(head, X(cols, tr), df.loc[tr, 'age'], cv=KFold(5, shuffle=True, random_state=0))
    head.fit(X(cols, tr), df.loc[tr, 'age'])
    df.loc[tr, f'pred {name}'] = oof
    row = {'model': f'{name} + ridge', 'Guys+HH': mae(oof, df.loc[tr, 'age'])}
    for s in ('eNKI',):
        k = (df.site == s).values
        p = head.predict(X(cols, k))
        df.loc[k, f'pred {name}'] = p
        row[s] = mae(p, df.loc[k, 'age'])
    table.append(row)
res = pd.DataFrame(table).set_index('model').round(2)
res.to_csv(OUT / 'transfer_head_mae.csv')
df.drop(columns=['gap', 'dense', 'bangf']).to_csv(OUT / 'transfer_head_predictions.csv', index=False)
log('MAE in years (Guys+HH = 5-fold out-of-fold; eNKI never used to train the heads):\n' + res.to_string())
log('caveat: BrainAgeNeXt was pretrained on openBHB (includes IXI) and NKI-RS - its rows are not held-out results')
(OUT / 'DONE.txt').write_text(f'{datetime.datetime.now()}\n' + res.to_string() + '\n')
log('ALL DONE. Results: outputs/transfer_head_mae.csv and outputs/transfer_head_predictions.csv')
release()